## 0 · The Challenge

> **The mission**: Riverside House must adapt the 361,821,120-parameter `HuggingFaceTB/SmolLM2-360M-Instruct` model locally without sending its confidential manuscript corpus to a public API.

**What we know so far:**

- SmolLM2 can already produce simple English fiction on Riverside's CPU.
- Riverside can test every candidate with the same three acceptance probes: catalog fluency, instruction compliance, and editor preference.
- **But the base model fails the first probe:** it has never seen Riverside's unpublished stories.

**What's blocking us:**
The model has never experienced Riverside's corpus. Giving it the manuscripts may improve continuation, but that only practices prose. If it later ignores a bounded editor request, that failure must determine the next training experience. If it follows the request but chooses a needlessly verbose answer, that remaining failure determines the final one.

**What this chapter unlocks:**
A failure-driven training path. We will expose one failed probe, apply the smallest data-objective change that addresses it, then rerun the probes before adding another technique.

# LLM Fine-Tuning Deep Dive, Part 1 of 3: What Should the Model Learn?

> **The story:** Riverside will change the model's training experience only when an observed failure justifies it: unfamiliar catalog prose motivates continued pretraining, ignored requests motivate SFT, and inferior choices among valid answers motivate preference learning.
>
> **Where you are:** The transformer chapters explained next-token prediction. Part 1 changes **what behavior the examples teach**; Part 2 changes **where the update is stored**; Part 3 asks **what the evidence supports**.
>
> **Terms:** a preference example contains one prompt, a chosen response, and a rejected response. DPO compares a trainable live policy with a frozen copy of the accepted SFT model.

## Riverside's Brief

Riverside House has eight unpublished novels in 225 chapter files, totaling about 570,000 words of chapter text. Manuscript text cannot leave the building, so every teaching run uses `HuggingFaceTB/SmolLM2-360M-Instruct` locally.

The immediate job is an editing assistant that can continue Riverside prose, obey a bounded request, and favor the response an editor would keep. Those are distinct behaviors, so each receives its own training signal and acceptance probe.

| Part | Question |
| --- | --- |
| 1 - this notebook | What training experience addresses the observed failure? |
| [2 - parameter strategy](02-llm-finetuning-parameter-techniques.ipynb) | How much model state must move, and what remains resident? |
| [3 - comparison and decision](03-llm-finetuning-comparison-and-decision.ipynb) | What does each result prove, and what can ship? |

The runnable checkpoints are not one mandatory ancestry chain: continued pretraining and SFT start from separate base models; DPO continues from the SFT adapter.

## Corpus and Setup

The committed `content/` directory contains the private teaching corpus. Run `setup.ps1`, select the `llm-tuning` kernel, and execute from a clean kernel. Checkpoints are written under `./checkpoints/` for Parts 2 and 3.

> **Boundary:** fine-tuning changes persistent behavior. A later retrieval chapter supplies current, citable corpus facts.

## Fine-Tuning Roadmap: Start Here

This map stays with the three-notebook arc. Read it vertically: Part 1 teaches **what behavior to learn**, Part 2 changes **how many parameters learn it**, and Part 3 decides **which evidence matters for each workload**.

```mermaid
flowchart TD
    Start["Starting point<br/>Base SmolLM2: fluent, but domain-blind"]

    subgraph Data["Part 1 - Current notebook: choose the learning objective"]
        direction TB
        C1["[Now] Concept 1<br/>Continued pretraining<br/>Teach catalog language and style"]
        C2["[Next] Concept 2<br/>SFT<br/>Teach instruction following"]
        C3["[Next] Concept 3<br/>DPO<br/>Teach editor preference"]
        C1 --> C2 --> C3
    end

    subgraph Params["Part 2 - Next notebook: choose the parameter strategy"]
        direction TB
        C4["Concept 4<br/>Full fine-tuning"]
        C5["Concept 5<br/>Partial freezing"]
        C6["Concept 6<br/>LoRA"]
        C7["Concept 7<br/>QLoRA + quantization"]
        C4 --> C5 --> C6 --> C7
    end

    Start --> C1
    C3 --> Saved["Part 1 checkpoint<br/>Three capability-focused artifacts"]
    Saved --> C4
    C7 --> Compare["Part 3<br/>Compare evidence and choose per workload"]
```

> The arrows are a learning sequence, not literal model ancestry. In the runnable examples, continued pretraining and SFT start from separate base models; DPO continues from the SFT adapter. The roadmap tracks questions answered, while checkpoint tables track the actual artifacts.

## Prerequisite Bridge: From Encoder-Decoder Attention to a Decoder-Only Assistant

The transformer foundations introduced three useful shapes: an **encoder** reads an entire input, a **decoder** predicts the next token while respecting a causal mask, and an **encoder-decoder** model lets a decoder attend to an encoded source through cross-attention. Riverside's assistant uses the decoder-only choice: at each turn, the user's instruction, any supplied scene, and the completion form one growing token sequence; causal self-attention lets each new token use everything to its left without seeing its own future.

| Foundation                | Role in this chapter                                              | Why Riverside needs it                                                                               |
| ------------------------- | ----------------------------------------------------------------- | ---------------------------------------------------------------------------------------------------- |
| Causal decoder            | `HuggingFaceTB/SmolLM2-360M-Instruct` predicts the next token      | It can continue prose and answer prompts from one left-to-right context                              |
| Training objective        | Labels say which next tokens should become more likely            | Continued pretraining, SFT, and DPO each change what Riverside teaches the same decoder              |
| Encoder / retrieval later | Encodes a query and passages for matching                         | It finds current, citable manuscript evidence instead of asking the generator to remember every fact |

So this notebook changes **how a decoder-only model behaves**. It does not turn the model into a dependable catalog lookup system: that next requirement leads to hybrid retrieval after the fine-tuning decision.

The underlying mechanics are owned by the prerequisite notebooks:

- [Transformers Part 12](../02-transformers/transformers.ipynb#part-12---the-causal-triangle-and-the-accumulation-tower) explains causal visibility and why each position cannot see its future.
- [Transformers Part 8](../02-transformers/transformers.ipynb#part-8---mini-language-model-training--inference) owns the decoder-only training loop, including the [per-position loss microscope](../02-transformers/transformers.ipynb#per-position-loss-one-sequence-many-lessons) and [one complete backward/update trace](../02-transformers/transformers.ipynb#one-backward-pass-many-token-lessons-one-update).
- [PyTorch RNN Bridge Part 4](../01-rnns/01-pytorch-rnn-bridge.ipynb#part-4--explicit-sequence-training-and-gradient-clipping) owns shifted sequence loss, backpropagation, and optimizer mechanics.
- [Encoder-Decoder Part 5](../03-encoder-decoder/encoder-decoder.ipynb#part-5--full-encoder-decoder-training) owns teacher-forced seq2seq training and cross-attention, with a [target-position update trace](../03-encoder-decoder/encoder-decoder.ipynb#teacher-forced-inner-mechanics-source-once-target-positions-together).

This notebook assumes those mechanics and focuses on the fine-tuning decision: **which examples and labels teach the behavior Riverside needs?**


## Three Failures, Three Training Signals

![Three fine-tuning data objectives: continued pretraining, supervised fine-tuning, and direct preference optimization](images/data-objectives-pipeline.png)

| Observed failure | Training experience | Technique | Success looks like |
| --- | --- | --- | --- |
| Catalog prose is generic or inconsistent | Predict the next token in raw Riverside text | **Continued pretraining** | Riverside continuations become less generic |
| The model continues a request instead of obeying it | Pair requests with desired responses | **SFT** | Task, format, and stopping rules are followed |
| Several responses are valid but not equally useful | Compare chosen and rejected responses | **DPO** | Editor-preferred responses gain ground |

The sequence is diagnostic, not compulsory. Stop when the required behavior passes its probe; do not add another objective merely because it exists.

> **Implementation preview:** the objective and parameter strategy are separate choices. This notebook uses full fine-tuning for the continued-pretraining demonstration, then small LoRA adapters for SFT and DPO so the runs fit local hardware. Treat LoRA here as a small trainable correction attached to a frozen base; Part 2 opens that black box and compares it with full and partial fine-tuning.

## Learning Route

1. Establish the unchanged base model and three acceptance probes.
2. Let catalog-fluency failure create the need for continued pretraining.
3. Let instruction non-compliance create the need for SFT and prompt masking.
4. Let competing valid answers create the need for preference data and DPO.
5. Compare the resulting checkpoints as behavior demonstrations, not a leaderboard.
6. Continue to Part 2 for parameter cost and Part 3 for workload evidence.

**Optional depth:** token-level mechanics now live in the prerequisite notebooks linked below. The production orchestration section remains a reference; skip it on a first practical pass and return when designing job boundaries.

In [ ]:
from pathlib import Path

# Resolve the notebook's own directory (content/ lives next to this notebook). VS Code's Jupyter
# kernels run with cwd = workspace root, not the notebook's folder, so __vsc_ipynb_file__ (which
# VS Code injects) is the reliable way to find it; __file__ covers plain .py execution.
try:
    _notebook_dir = Path(__vsc_ipynb_file__).parent  # type: ignore[name-defined]
except NameError:
    try:
        _notebook_dir = Path(__file__).parent
    except NameError:
        _notebook_dir = Path.cwd()

CONTENT_DIR = _notebook_dir / "content"
if not CONTENT_DIR.exists():
    raise FileNotFoundError(
        f"Could not find the corpus at {CONTENT_DIR}. Open and run this notebook from its own "
        "location in the repo (learning/genai/04-llm/) so its content/ folder resolves correctly."
    )

print(f"Content directory: {CONTENT_DIR.absolute()}")

# Novel directory mappings (keys are shorthand aliases, values are actual directory names)
NOVELS = {
    "scifi": "the-weight-of-distant-light",  # 40 chapters
    "fantasy": "the-tidebound-accord",  # 33 chapters
    "mystery": "the-cartographers-cipher",  # 21 chapters
    "historical": "the-silk-merchants-daughter",  # 23 chapters
    "cyberpunk": "neural-drift",  # 24 chapters
    "horror": "the-hollow-beneath",  # 28 chapters
    "literary": "the-weight-of-tides",  # 28 chapters
    "everglades": "the-everglades-cipher",  # 28 chapters
}


def load_corpus_paragraphs(novels=None, max_chapters=10, min_len=200):
    """Load paragraphs from selected novels, or the complete corpus when novels is None.

    Args:
        novels: Novel aliases to load, or None for every directory in NOVELS.
        max_chapters: Maximum chapters per novel, or None for every chapter.
        min_len: Skip paragraphs shorter than this many characters.

    Returns:
        List of paragraph strings from all requested novels.
    """
    if novels is None:
        novels = list(NOVELS.keys())

    paragraphs = []
    for alias in novels:
        novel_dir = NOVELS.get(alias)
        if not novel_dir:
            print(f"Warning: unknown novel alias '{alias}', skipping")
            continue

        novel_path = CONTENT_DIR / novel_dir
        if not novel_path.exists():
            print(f"Warning: directory {novel_path} not found, skipping")
            continue

        chapter_files = sorted(novel_path.glob("chapter-*.txt"))
        if max_chapters is not None:
            chapter_files = chapter_files[:max_chapters]
        for path in chapter_files:
            text = path.read_text(encoding="utf-8")
            for para in text.split("\n\n"):
                para = para.strip().replace("\n", " ")
                if len(para) >= min_len:
                    paragraphs.append(para)

    return paragraphs


# Sample from 4 genres to show multi-genre paragraph diversity
sample_paragraphs = load_corpus_paragraphs(
    novels=["scifi", "fantasy", "mystery", "horror"], max_chapters=2
)
print(f"Loaded {len(sample_paragraphs)} sample paragraphs from 4 novels. First one:\n")
print(sample_paragraphs[20][:421], "...")

## Baseline: Let the Model Fail Before Naming a Technique

Riverside will reuse three probes after every training stage:

| Probe | Request | Failure signal |
| --- | --- | --- |
| Catalog fluency | Continue a passage containing Riverside-only names and relationships | Generic continuation or invented story facts |
| Instruction compliance | `Answer in one sentence and stop.` | Restates the request, rambles, or violates the format |
| Editor preference | Compare two valid answers to the same request | No consistent reason to favor the concise, useful answer |

Start with the catalog-fluency probe. The base model has never seen Riverside's manuscripts, so a fluent answer is not evidence of knowledge. It can only guess from names in the prompt.

That gives us the first failure to fix:

> The model knows how English works, but Riverside language is still surprising to it.

---

### Setting Up the Shared Baseline

All three stages use the same base checkpoint and fixed prompts. Keeping an untouched `base_model` gives every later comparison a real before state rather than a remembered sample.

### Pin a CPU-Friendly Base Model

Every comparison needs one unchanged starting point. Riverside uses `HuggingFaceTB/SmolLM2-360M-Instruct`, an instruction-tuned Llama causal decoder with 361,821,120 parameters, 32 decoder blocks, and hidden size 960.

Comparing different models and their suitability on a GPU-enabled device is outside the scope of this notebook. SmolLM2-360M is compact by modern LLM standards while retaining a credible general-language baseline. Changing `MODEL_NAME` invalidates checkpoints from another architecture.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "HuggingFaceTB/SmolLM2-360M-Instruct"
SYSTEM_PROMPT = "You are Riverside House's concise fiction-writing assistant."
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]
DEMO_TRAIN_STEPS = 10
DEMO_DPO_STEPS = 10

### Choosing a Device: GPU if Available, CPU Otherwise

`device` tells PyTorch where tensors and model weights should physically live. We need this because
every tensor operation in this notebook (forward pass, backward pass, `.generate()`) has to run on
the same device as the weights, or PyTorch raises a device-mismatch error.

`torch.cuda.is_available()` checks for a usable NVIDIA GPU + CUDA driver; if none is found, we fall
back to `"cpu"` so the notebook still runs end-to-end (just slower) on a laptop with no dedicated
GPU -- Riverside's actual situation.


> **PyTorch → Keras:** `torch.cuda.is_available()` — checks whether a CUDA-capable GPU is visible to PyTorch and returns a bool; the result picks the `device` string (`"cuda"` or `"cpu"`) that every tensor and model call below is pinned to via `.to(device)`. **Keras/TF equivalent:** `tf.config.list_physical_devices('GPU')` — TensorFlow auto-places ops on any visible GPU without needing an explicit device string threaded through the code, so most Keras code skips this check entirely; `tf.device(...)` exists for the rare case you want to force placement.

In [ ]:
device = (
    "cuda" if torch.cuda.is_available() else "cpu"
)  # detects whether a GPU is available
print(f"Using device: {device}")

### Loading the Tokenizer and Instruction Format

The checkpoint uses SmolLM2's 49,152-token vocabulary and built-in chat template. `<|im_end|>` is both the padding and end-of-message token. Whitespace and punctuation can change token boundaries, so the following cells inspect actual IDs rather than assuming word-level tokens.

SFT, DPO, and instruction evaluation all use the tokenizer's native system/user/assistant serialization:

```text
<|im_start|>system
...
<|im_end|>
<|im_start|>user
...
<|im_end|>
<|im_start|>assistant
...
<|im_end|>
```

For batching, padded labels are masked with `-100`, so padding contributes no training loss.


> **PyTorch → Keras:** `AutoTokenizer.from_pretrained(MODEL_NAME)` loads the same checkpoint-matched
tokenizer for either framework. Tokenization and explicit instruction-text serialization are framework-agnostic; the split
between PyTorch and TensorFlow begins only when the resulting arrays are converted to framework tensors.

In [ ]:
# SmolLM2 provides the chat template used consistently by SFT, DPO, and evaluation.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def render_instruction(instruction, response=None):
    """Render one native SmolLM2 chat contract for SFT, DPO, and evaluation."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": instruction.strip()},
    ]
    if response is None:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    messages.append({"role": "assistant", "content": response.strip()})
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )


print(f"Tokenizer vocabulary size: {len(tokenizer):,}")
print(f"Pad token: {tokenizer.pad_token!r} (id={tokenizer.pad_token_id})")
print(f"EOS token: {tokenizer.eos_token!r} (id={tokenizer.eos_token_id})")
print(render_instruction("Continue this sentence: The signal arrived")[:240])


### Seeing the Vocabulary in Action

Byte-level BPE can represent arbitrary text, but common spans receive compact tokens while rare names or
unusual Unicode sequences split into several pieces. Whitespace is context: tokenizing `"signal"` and
`" signal"` can produce different IDs because a space may be merged with neighboring bytes.

The code below prints IDs, raw tokenizer tokens, and decoded pieces for each example. Decoding each ID is
the portable way to make spaces and newlines visible; raw token strings are implementation details and
should not be treated as a universal notation.


> **PyTorch → Keras:** `tokenizer.encode(word)` / `tokenizer.convert_ids_to_tokens(ids)` — converts raw text to integer token IDs (and back to readable BPE-piece strings) using the framework-agnostic tokenizer loaded above; no tensors are created yet, just plain Python lists. **Keras/TF equivalent:** identical call — `AutoTokenizer` isn't PyTorch- or TF-specific, so a Keras/TF version of this notebook would use this exact same code; only the downstream model call (`TFAutoModelForCausalLM` vs. `AutoModelForCausalLM`) would differ.

In [ ]:
# One example each of a noun, proper noun, verb, and adjective -- all pulled from Riverside's own
# sci-fi opening line, so these are words this notebook already leans on elsewhere.
example_words = {
    "noun": "signal",
    "proper noun": "Aria",
    "verb": "stared",
    "adjective": "distant",
}

for part_of_speech, word in example_words.items():
    ids_alone = tokenizer.encode(word)  # tokenize the word standalone (no leading space)
    ids_mid_sentence = tokenizer.encode(" " + word)  # tokenize as it would appear mid-sentence
    print(f"{part_of_speech.upper()}: {word!r}")
    print(
        f"  as the first word of a text  : ids={ids_alone}  "
        f"tokens={tokenizer.convert_ids_to_tokens(ids_alone)}"
    )
    print(
        f"  mid-sentence (' {word}')".ljust(31) + f": ids={ids_mid_sentence}  "
        f"tokens={tokenizer.convert_ids_to_tokens(ids_mid_sentence)}"
    )
    print()

print(
    "'\u0120' at the start of a token marks a leading space -- it's why the same word can tokenize "
    "differently depending on where it appears in a sentence."
)


> **You may wonder:** since `Aria` splits into two tokens and appears constantly in this corpus, why
> not just train the tokenizer on Riverside's own text and merge it into a dedicated token? Two
> reasons this is out of scope for fine-tuning: extending the vocabulary adds a new, untrained row
> to the embedding matrix and output head, and filling that row in with a meaningful representation
> is itself a training problem, not something fine-tuning does for free. And splitting `Aria` into
> two tokens doesn't stop the model from learning what it means -- it can still learn to associate
> that two-token pattern with everything fine-tuning teaches it about her; it just costs two sequence
> positions instead of one, a small efficiency tax, not a correctness problem. The actual gap the
> rest of this notebook closes is that the model has never seen who Aria Voss is, not how her name
> happens to be tokenized.



> **A related question:** what happens with a word the tokenizer has rarely encountered? Byte-level
> BPE does not need an unknown-word vocabulary entry: when no longer merge matches, it falls back to smaller
> byte-derived pieces. An uncommon name such as `Itzpapalotl` therefore remains representable, although it
> usually consumes more tokens than a frequent word. Fine-tuning can improve how the model uses that sequence,
> but it does not add a new vocabulary row unless the tokenizer and embedding matrix are explicitly resized.


### Loading the Base Model

`base_model` is the actual pretrained neural network -- a checkpoint-defined number of real weights downloaded from the
Hugging Face hub, moved onto whichever device we resolved above via `.to(device)`. This untouched
checkpoint is the "before" every fine-tuning technique in this notebook is compared against.


> **PyTorch → Keras:** `AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)` — downloads the weights selected by `MODEL_NAME` into a PyTorch `nn.Module` and moves every parameter tensor onto `device` (CPU or GPU) in place. **Keras/TF equivalent:** `TFAutoModelForCausalLM.from_pretrained(MODEL_NAME)` — loads the same checkpoint into a `tf.keras.Model` instead; TensorFlow doesn't need an explicit `.to(device)` call since ops are placed on available devices automatically (or via a `tf.device(...)` context).

In [ ]:
# Download the lightweight base model and keep its architecture facts runtime-derived.
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
model_parameter_count = sum(parameter.numel() for parameter in base_model.parameters())
decoder_blocks = base_model.model.layers
n_blocks = len(decoder_blocks)
hidden_size = base_model.config.hidden_size
print(
    f"Loaded {MODEL_NAME}: {model_parameter_count:,} parameters, "
    f"{n_blocks} decoder blocks, hidden size {hidden_size}."
)

### A Fixed Test Prompt for Before/After Comparisons

`PROMPT` is the one fixed test sentence reused throughout the notebook so "before" vs. "after"
fine-tuning comparisons are always apples-to-apples. It's pulled straight from the sci-fi corpus so
a model that has actually absorbed the catalog has a real chance of continuing it in-world.


In [ ]:
PROMPT = "Aria Voss stared at the signal counting itself out in prime numbers and"  # from the sci-fi corpus

### A Reusable `generate()` Helper

Hugging Face returns the prompt and completion in one token sequence. The helper records the prompt length, slices `output[prompt_length:]`, and decodes only the new tokens so every comparison shows the model's actual continuation.

It also calls `.strip()` because the first generated token may carry leading whitespace. All later candidates use this same helper, so output formatting cannot masquerade as a model difference.

> **PyTorch → Keras:** `model.eval()` / `torch.no_grad()` / `model.generate()` — `.eval()` switches dropout/batchnorm-style layers to inference mode, `torch.no_grad()` disables gradient tracking to save memory during inference, and `.generate()` runs HuggingFace's autoregressive sampling loop (nucleus sampling here via `top_p`/`temperature`). **Keras/TF equivalent:** `TFAutoModelForCausalLM.generate()` — the same HuggingFace `.generate()` API exists on TF models with identical sampling arguments; TF's analog of "eval mode" is passing `training=False` (implicit inside `.generate()`), and there's no separate "no_grad" context since calling a `tf.keras.Model` outside a `GradientTape` block already skips gradient recording.

In [ ]:
def generate(model, prompt, max_new_tokens=60):
    """Generate a text continuation for *prompt*.

    Returns **only the newly generated tokens** (prompt is stripped), so every
    print(generate(...)) call in this notebook shows the model's actual output
    without echoing the input back.

    Parameters
    ----------
    model : PreTrainedModel or PeftModel
        Any HuggingFace causal-LM model (base model, LoRA adapter, DPO policy …)
    prompt : str
        The input text passed to the model.
    max_new_tokens : int
        Hard cap on how many new tokens to generate after the prompt ends.
        The model can stop earlier if it samples the EOS token.

    Notes
    -----
    A real example from this notebook's own `PROMPT` (15 tokens) makes both
    lines concrete. Asking for `max_new_tokens=15` returns `out` with shape
    `(1, 30)` -- the 15 prompt tokens plus 15 new ones, concatenated. Decoding
    all 30 without slicing prints the prompt right back before the answer:

        'Aria Voss stared at the signal counting itself out in prime numbers
         and began to ponder the question, what was it that she had to do?'

    `out[0][prompt_len:]` (`prompt_len = 15` here) drops the first 15 tokens so
    only the new continuation gets decoded. But decoding *just* those 15 new
    tokens gives:

        ' began to ponder the question, what was it that she had to do?'

    -- note the stray leading space: the decoded continuation can begin with whitespace carried by its first
    token, so the raw decoded string may start with a space. `.strip()`
    removes it, along with any trailing whitespace/newlines near the end.
    """
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt").to(
        device
    )  # use the same tokenizer to tokenize the prompt and convert it to tensor
    prompt_len = inputs["input_ids"].shape[1]  # track where the prompt ends
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,  # stochastic → varied output
            top_p=0.9,  # nucleus sampling: top 90% mass
            temperature=0.8,  # soften distribution slightly
            pad_token_id=tokenizer.pad_token_id,
        )

    # out[0] shape: (prompt_len + new_tokens,)
    # Slice from prompt_len onward to get ONLY the model's continuation
    completion = tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True).strip()
    return (
        completion
        if completion
        else "[model stopped immediately — sampled EOS as first token]"
    )


print("=== Baseline (no fine-tuning) — model continuation only ===")
print(f"Prompt    : {PROMPT}")
print(f"Completion: {generate(base_model, PROMPT, 20)}")

### Code Walkthrough: Shared Setup

**1. Native pad/EOS token**

For SmolLM2, `<|im_end|>` serves as both the padding and EOS token. Padding labels are still replaced with `-100` before loss is computed.

**2. Runtime-derived architecture**

`AutoModelForCausalLM.from_pretrained(MODEL_NAME)` loads the checkpoint. Parameter count, decoder-block count, and hidden width come from the loaded object rather than hard-coded prose.

**3. Native chat formatting**

`render_instruction()` uses SmolLM2's native `system`/`user`/`assistant` chat template for SFT examples, DPO pairs, and instruction evaluation. Plain causal continuation, including continued pretraining, remains unformatted.

## Test Prompts: Define Success Before Training

A fluent continuation is not enough; the base model is already fluent. A successful Riverside adaptation must use story-specific entities coherently without copying the prompt or inventing another world.

Use three probe families:

| Probe | What it checks | Failure signal |
| --- | --- | --- |
| Character and setting | Catalog-specific relationships | Generic roles, places, or invented lore |
| Genre continuation | Prose and narrative behavior | Fluent text in the wrong voice or genre |
| Cross-novel vocabulary | Breadth beyond one manuscript | Improvement only on memorized phrases |

For example, after `Aria Voss checked the Meridian's Promise status panel and`, an adapted continuation should remain aboard Riverside's ship and use established relationships; mentioning the name alone does not count.

The executable `TEST_PROMPTS` dictionary in the next cell is the source of truth for the full multi-genre fixture set. Keeping the prompts in code avoids maintaining the same catalog twice.

In [ ]:
# Automated test runner: compare baseline vs fine-tuned on corpus-specific prompts
TEST_PROMPTS = {

    # Sci-fi — The Weight of Distant Light
    "scifi_character": "Aria Voss checked the Meridian's Promise status panel and",
    "scifi_keeper": "The Keeper's consciousness flickered through node seventeen as",

    # Fantasy — The Tidebound Accord
    "fantasy_magic": "Kerra Valmont felt all five tides simultaneously—water, wind, stone, flame, and void—as",
    "fantasy_hollow_king": "The Hollow King's followers, called the Hollowed, began to gather when",

    # Mystery — The Cartographer's Cipher
    "mystery_conspiracy": "The six founding families—Ashmont, Thorne, Blackwell, Winters, Kahale, and Mordecai—",
    "mystery_elena": "Elena Voss studied the 1879 survey map and realized the Ashmont Trust",

    # Historical — The Silk Merchant's Daughter
    "historical_setting": "Wei Lian's jade phoenix pendant caught the morning light in Chang'an as",
    "historical_silk_road": "The delegation crossed the Taklamakan desert and Wei Lian noted in her ledger",

    # Cyberpunk — Neural Drift
    "cyberpunk_tech": "Kai Chen adjusted the neurorig and prepared to extract the memory backup from",
    "cyberpunk_project": "In the Lower Stacks of Neo-Shanghai, the stolen neural backups from Project Drift",

    # Horror — The Hollow Beneath
    "horror_atmosphere": "Eleanor Vance sealed the cellar door at sunset, knowing that Blackwood Manor",
    "horror_chambers": "The tenth chamber of the Hollow pulsed with a light that had no source, and Eleanor",

    # Literary — The Weight of Tides
    "literary_marine": "Claire Merritt opened her father's blue folder and read the July 12, 1975 entry about",
    "literary_contact": "The Observer surfaced near Whitehead Island and Claire understood for the first time that",
}


def test_corpus_knowledge(model, test_prompts=TEST_PROMPTS, max_new_tokens=50):
    """Run all test prompts and return results dict for comparison."""
    results = {}
    prompts = {}
    for key, prompt in test_prompts.items():
        prompts[key] = prompt
        results[key] = generate(model, prompt, max_new_tokens=max_new_tokens)  # this model's continuation for each prompt
    return results, prompts


# Run baseline tests (will show generic, off-corpus continuations)
print(
    "=== BASELINE MODEL (no fine-tuning) - should produce generic continuations ===\n"
)
baseline_results, prompts = test_corpus_knowledge(base_model)
for key, output in baseline_results.items():
    print(f"[{key}]")
    print(prompts[key] + " .... " + output[:200] + "...\n")


Notice the output has no awareness of Aria Voss (from _The Weight of Distant Light_), the _Meridian's
Promise_, the Lantern, or any of the other characters/worlds across Riverside's eight novels -- it is fluent
English but a generic, unrelated continuation. This is exactly the gap fine-tuning closes.

Per the "What 'success' actually looks like" example above, a model that had genuinely absorbed this
corpus would instead keep Aria aboard the Meridian's Promise, in her actual role, referencing the
Under-Hold or the Lantern instead of inventing an unrelated ship and crew. This notebook doesn't
re-run `test_corpus_knowledge()` on a fine-tuned checkpoint (fine-tuning a fresh model per test prompt
would multiply the compute cost of every section below), but the Ablation Study near the end trains
and compares checkpoints on a closely related Aria Voss / Meridian's Promise prompt, so you can see
the real before/after side by side.


### Transformer Mechanics Live Upstream

A causal-LM training batch still follows one compact contract: token IDs enter the decoder, each position predicts the next token, padding labels use `-100`, and backpropagation updates whichever parameters remain trainable.

This chapter does not re-derive that contract. Use the prerequisite links above for causal masks, per-position cross-entropy, gradient flow, and optimizer updates. From here onward, every code path earns its place by answering a fine-tuning question:

- Which Riverside text becomes continued-pretraining data?
- Which prompt tokens must SFT hide from the loss?
- Which chosen/rejected pairs express editor preference?
- Which checkpoint and evaluation evidence supports promotion?

The next section begins at that fine-tuning-specific boundary.


In [ ]:
# Shared analysis imports used by later fine-tuning diagnostics.
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib.patches import Patch, Rectangle
from IPython.display import display, HTML
import warnings

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 100, "font.size": 10})
sns.set_theme(style="whitegrid", palette="muted")

print("Shared analysis libraries loaded.")


---

## The Fine-Tuning Journey: Let Each Failure Choose the Next Objective

```mermaid
flowchart TD
    A["Base model<br/>fluent, catalog-blind"] -->|"generic Riverside prose"| B["Continued pretraining"]
    B -->|"continues requests instead of obeying them"| C["SFT"]
    C -->|"valid answer, wrong editorial choice"| D["DPO or another preference method"]
```

This is a diagnostic sequence, not mandatory checkpoint ancestry. Stop as soon as Riverside's required behavior passes its acceptance probes.

Part 1 changes the **training experience**. Part 2 separately asks whether full fine-tuning, freezing, LoRA, or QLoRA can carry that experience within Riverside's hardware and release constraints.

## Concept 1: Continued Pretraining - Make Riverside Prose Less Surprising

The baseline failed on Riverside-only language because those names, relationships, and stylistic patterns were absent from its training experience.

**Minimal fix:** keep the original next-token objective, but continue training on raw Riverside paragraphs. There are no instructions or preference labels yet; the model simply predicts the next manuscript token. This is **continued pretraining**, also called domain-adaptive pretraining.

| What this experience can teach | What it cannot teach |
| --- | --- |
| Domain vocabulary, recurring entities, prose patterns | How to obey a user request |
| Which continuations resemble Riverside text | When to stop or return a required format |
| A better prior for later adaptation | Which of two acceptable answers an editor prefers |

The runnable example updates all model weights so the learning signal is easy to inspect. Part 2 will challenge that expensive parameter choice.

**Checkpoint after training:** rerun the catalog-fluency probe, then issue a bounded request such as `Answer in one sentence and stop.` If the continuation becomes more Riverside-like but the model still treats the request as text to continue, continued pretraining worked and exposed the next blocker.

> **Bridge to SFT:** the model has practiced Riverside prose, not the interaction contract of an assistant. The next section exists because knowing the domain and following an instruction are different behaviors.

### Code Walkthrough: `tokenize_causal()` — Preparing Text for Next-Token Prediction

This is the first point in the notebook where we actually need to convert raw paragraph strings into
the fixed-length integer tensors a transformer consumes, so this is where `tokenize_causal()` gets
defined, right before the `dataset.map(...)` call that needs it. Every later stage that trains on
plain continuation text (partial freezing and LoRA continued pretraining, further down) reuses this
exact same function; the instruction-tuning and DPO sections swap in a response-masked variant
instead, since those need to hide the prompt from the loss.

```python
def tokenize_causal(examples, tokenizer, max_length=64):
    tokens = tokenizer(
        examples["text"], truncation=True, padding="max_length", max_length=max_length
    )
    labels = [
        [(tok if mask == 1 else -100) for tok, mask in zip(ids, attn)]
        for ids, attn in zip(tokens["input_ids"], tokens["attention_mask"])
    ]
    tokens["labels"] = labels
    return tokens
```

**Arguments:**

| Argument     | Type                  | Purpose                                                                                                                                                                                                                                        |
| ------------ | --------------------- | ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| `examples`   | `dict` (HF batch)     | A batch of examples with a `"text"` column — here, the raw paragraph strings in `non_inst_paragraphs`, wrapped in a `Dataset`.                                                                                                                 |
| `tokenizer`  | `PreTrainedTokenizer` | The tokenizer loaded in the baseline cell above (`AutoTokenizer.from_pretrained(MODEL_NAME)`). Passed in explicitly rather than closed over, so the same function works unchanged no matter which model/tokenizer this notebook is pointed at. |
| `max_length` | `int`, default `64`  | Hard cap on sequence length. Longer paragraphs are truncated; shorter ones are padded up to this length so every example in a batch has the same shape.                                                                                        |

**What it returns:** the usual tokenizer output (`input_ids`, `attention_mask`) plus a `labels` key,
which HuggingFace's `Trainer` requires to compute the causal-LM loss. `labels` starts as a copy of
`input_ids`, then every padding position (where `attention_mask == 0`) is overwritten with `-100` —
PyTorch's `CrossEntropyLoss` convention for "ignore this position." Without that mask, the model would
waste training signal learning to predict padding tokens instead of real text.

It's called in the cell below as `dataset.map(lambda ex: tokenize_causal(ex, tokenizer), batched=True, remove_columns=["text"])`.
`batched=True` is what makes `examples` a dict-of-lists (every text in the batch at once) instead of a
single example, which is why the list comprehension inside zips over `tokens["input_ids"]` rather than
indexing a single sequence.


### Optional Depth: Long Documents, Truncation, and Packing

Plain `truncation=True` is a paper cutter: without overflow handling, an 800-token example capped at 512 contributes only its first 512 tokens.

This notebook's `tokenize_causal()` also sets `return_overflowing_tokens=True`, so a long paragraph becomes multiple fixed-length chunks instead of silently losing its tail. The final short chunk is padded and its padding labels are masked.

| Training type | Common strategy | Trade-off |
| --- | --- | --- |
| SFT | Truncate or separately budget prompt and completion | Preserves pair structure, but an overlong response may still lose its tail |
| Continued pretraining | Overflow chunks or pack documents into fixed blocks | Preserves more text, but block boundaries weaken cross-boundary context |

```text
BLOCK A: [ Token 0 ... Token 127 ]
BLOCK B: [ Token 128 ... ]  <- attention starts again here
```

The first tokens in Block B cannot attend to Block A even when they continue the same paragraph. Production pipelines may use document-aware packing, block-diagonal attention, best-fit grouping, or local overlap to manage that trade-off.

For this teaching run, overflow chunks keep every paragraph tail visible while preserving a simple fixed-length loss mask.

> **PyTorch → Keras:** `from datasets import Dataset` / `from transformers import Trainer, TrainingArguments` / `dataset.map(...)` — HuggingFace's `Dataset.map()` applies `tokenize_causal()` to every example (batched, for speed), producing the `input_ids`/`attention_mask`/`labels` columns that `Trainer` (a full PyTorch training-loop wrapper: batching, forward/backward, optimizer step) consumes next. **Keras/TF equivalent:** `tf.data.Dataset.map(...)` + `model.fit(...)` — a Keras version would build a `tf.data.Dataset` pipeline with the same `.map()` call and then call the standard `model.fit(dataset, epochs=...)` in place of HuggingFace's `Trainer` (or use `TFAutoModelForCausalLM` with HuggingFace's own `Trainer`, which wraps `model.fit` under the hood).

In [ ]:
from datasets import Dataset
from transformers import Trainer, TrainingArguments

non_inst_paragraphs = load_corpus_paragraphs(
    novels=["scifi", "fantasy", "mystery"],
    max_chapters=None,  # None = all available chapters per novel
)
print(
    f"Loaded {len(non_inst_paragraphs)} paragraphs from 3 novels for continued-pretraining demo"
)

non_inst_dataset = Dataset.from_dict({"text": non_inst_paragraphs})  # wrap the paragraph list in a HF Dataset


def tokenize_causal(examples, tokenizer, max_length=64):
    """Standard next-token-prediction tokenization: labels = input_ids, with padding
    positions masked out (-100) so the loss ignores them."""
    tokens = tokenizer(
        examples["text"], truncation=True, padding="max_length", max_length=max_length, return_overflowing_tokens=True
    )
    # return_overflowing_tokens=True ensures that long texts are split into multiple chunks, each of max_length tokens
    # The last chunk with fewer than max_length tokens will still be included, padded to max_length
    # This ensures tokenizer is never dropping text due to length constraints

    # Copy input ids into labels, replacing padding positions with -100 so the loss skips them
    labels = [
        [(tok if mask == 1 else -100) for tok, mask in zip(ids, attn)]
        for ids, attn in zip(tokens["input_ids"], tokens["attention_mask"])
    ]
    tokens["labels"] = labels
    return tokens


non_inst_tokenized = non_inst_dataset.map(
    lambda ex: tokenize_causal(ex, tokenizer), batched=True, remove_columns=["text"]
)  # apply tokenize_causal across the whole dataset in batches, dropping the raw text column


Data's ready. Now load a **fresh, untouched copy** of `HuggingFaceTB/SmolLM2-360M-Instruct` to actually fine-tune -- kept
separate from `base_model` so `base_model` stays the permanent "before" snapshot every later
comparison in this notebook relies on.


> **PyTorch → Keras:** `AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)` / `p.numel()` — loads a second, independent copy of `HuggingFaceTB/SmolLM2-360M-Instruct` (kept separate from `base_model` so the original stays an untouched "before" snapshot) and `.numel()` counts the total scalar elements in each parameter tensor to report the full trainable-parameter count. **Keras/TF equivalent:** `TFAutoModelForCausalLM.from_pretrained(MODEL_NAME)` / `model.count_params()` — Keras models expose a built-in `count_params()` method that sums all trainable + non-trainable weight sizes in one call, instead of manually summing `p.numel()` over every parameter.

In [ ]:
import gc

# Make this cell safe to rerun without retaining the previous full-FT model through its Trainer.
if "trainer_full" in globals():
    del trainer_full
if "full_ft_model" in globals():
    del full_ft_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

full_ft_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)  # separate fresh copy dedicated to full fine-tuning
print(
    f"Loaded a fresh {MODEL_NAME} to fully fine-tune: "
    f"{sum(p.numel() for p in full_ft_model.parameters()):,} parameters, all trainable."
)


### Configuring and Running the Trainer

`TrainingArguments` + `Trainer` is HuggingFace's standard training loop -- it handles batching, the forward/backward pass, and the optimizer step described earlier in this notebook, so we do not write that loop by hand. `DEMO_TRAIN_STEPS = 10` and `learning_rate=5e-5` keep this CPU demonstration bounded; a real Riverside training run would choose its budget from measured convergence. `trainer_full.train()` runs all ten real optimizer steps used by every later comparison in this notebook.

> **PyTorch → Keras:** `TrainingArguments(...)` / `Trainer(model=..., args=..., train_dataset=...)` / `trainer_full.train()` / `full_ft_model.save_pretrained(...)` — configures and runs HuggingFace's full PyTorch training loop (batching, forward/backward passes, optimizer steps, logging) in one `.train()` call, then serializes the fine-tuned weights + config to disk. **Keras/TF equivalent:** `model.compile(optimizer=..., loss=...)` + `model.fit(dataset, epochs=...)` — the direct Keras analog of configuring + running training; `save_pretrained(...)` has an identically-named method on `TFPreTrainedModel` subclasses, so the checkpoint-saving line itself would be unchanged in a TF version.

In [ ]:
# Configure a short training run (max_steps kept small for CPU-friendly demo purposes)
training_args_full = TrainingArguments(
    output_dir="./checkpoints/non-instruction-full",
    per_device_train_batch_size=1,
    # number of steps during gradient descent, kept small for CPU-friendly demo purposes, in practice this would be around several thousand
    max_steps=DEMO_TRAIN_STEPS,
    logging_steps=1,
    save_strategy="no",
    learning_rate=5e-5,
    report_to="none",
)

# Wrap the model, config, and tokenized dataset in a Trainer and run the actual training loop
trainer_full = Trainer(
    model=full_ft_model, args=training_args_full, train_dataset=non_inst_tokenized
)
trainer_full.train()
full_ft_model.save_pretrained("./checkpoints/non-instruction-full")  # persist weights to disk for later reload
print("Saved continued-pretraining (full fine-tune) checkpoint.")


### What the Fully Fine-Tuned Model Actually Generates

Training loss shows that the model updated, but it does not show what changed at inference time. The next cell runs the same three held-out health-check prompts through both the untouched base model and the newly trained checkpoint. This makes domain adaptation, general-language retention, and possible memorization visible rather than inferred from the loss curve alone.

In [ ]:
# Compare real continuations before and after full continued pretraining.
full_training_prompts = {
    "Domain knowledge": "Aria Voss checked the Meridian's Promise and",
    "General knowledge retention": "The capital of France is",
    "Novel domain generalization": "In the Under-Hold, the rebels gathered and",
}

for label, prompt in full_training_prompts.items():
    print(f"=== {label} ===")
    print(f"Prompt : {prompt!r}")

    # manual seed controls which random numbers
    # are used for the initial values of the model's parameters.
    # a random seed generates a fixed set of random numbers that can be reproduced across runs
    # this ensures that the generated outputs are consistent and comparable across runs
    torch.manual_seed(42)
    print(f"Base   : {generate(base_model, prompt)}")
    torch.manual_seed(42)
    print(f"Trained: {generate(full_ft_model, prompt)}")
    print()

### Weight Movement Layer by Layer

A single aggregate can hide where full fine-tuning changed the network. The cell below samples the same number of individual absolute weight deltas from every transformer block and gives **each block its own panel**.

Figures contain at most 10 panels, so the runtime-reported decoder blocks are paginated automatically. Every panel uses the same y-axis scale: a quiet block therefore cannot look as active as a strongly moving block merely because its axis was automatically rescaled.

> **What to look for:** Compare the mean and maximum in each panel title, then inspect the shape. Long regions near zero mean many sampled weights barely moved; spikes identify sampled weights with larger changes. These are sampled magnitudes, not a claim that one block alone stores the learned behavior.

In [ ]:
# Load the saved checkpoint (full_ft_model was freed above; reload from disk for the layer panels)
# Snapshot the pre-fine-tune weights (base_model is the permanent "before" snapshot), so
# we can diff against them below.
import gc

base_state = dict(base_model.named_parameters())
ft_trace_model = AutoModelForCausalLM.from_pretrained(
    "./checkpoints/non-instruction-full"
).to("cpu")

SAMPLES_PER_BLOCK = 500  # weights sampled per transformer block
PANELS_PER_FIGURE = 10
N_COLUMNS = 2

all_deltas = []
for block_i in range(len(base_model.model.layers)):  # walk every transformer block in order
    block_deltas = []
    for name, parameter in ft_trace_model.named_parameters():
        if f"model.layers.{block_i}." in name:  # only this block's own parameters
            delta = (
                parameter.data.cpu() - base_state[name].data.cpu()
            ).abs().flatten()  # absolute weight movement vs. the untouched base checkpoint
            block_deltas.append(delta)
    if block_deltas:
        combined = torch.cat(block_deltas)
        stride = max(1, len(combined) // SAMPLES_PER_BLOCK)  # even subsampling so every block plots the same count
        sampled = combined[::stride][:SAMPLES_PER_BLOCK].float().numpy()
        all_deltas.append(sampled)

# Use one shared scale across every figure. Without this, a quiet block could look as active as
# the block with the largest movement simply because Matplotlib rescaled its panel.
global_ymax = max(float(block.max()) for block in all_deltas)
y_limit = global_ymax * 1.05 if global_ymax > 0 else 1e-9

for page_start in range(0, len(all_deltas), PANELS_PER_FIGURE):  # paginate blocks across multiple figures
    page = all_deltas[page_start : page_start + PANELS_PER_FIGURE]
    n_rows = (len(page) + N_COLUMNS - 1) // N_COLUMNS

    fig, axes = plt.subplots(
        n_rows,
        N_COLUMNS,
        figsize=(14, 2.6 * n_rows),
        sharex=True,
        sharey=True,
        squeeze=False,
    )
    axes = axes.ravel()

    for panel_i, block_arr in enumerate(page):  # one subplot per transformer block on this page
        block_i = page_start + panel_i
        weight_indices = np.arange(len(block_arr))
        axis = axes[panel_i]

        axis.plot(weight_indices, block_arr, linewidth=0.7, color="steelblue")
        axis.fill_between(weight_indices, block_arr, alpha=0.12, color="steelblue")
        axis.set_title(
            f"Transformer block {block_i}  "
            f"(mean={block_arr.mean():.2e}, max={block_arr.max():.2e})",
            fontsize=9,
        )
        axis.set_ylim(0, y_limit)
        axis.grid(alpha=0.2, axis="y")

    for unused_axis in axes[len(page) :]:
        unused_axis.set_visible(False)  # hide any empty grid cells on the last page

    page_end = page_start + len(page) - 1
    fig.suptitle(
        f"Full Fine-Tuning Weight Movement: Blocks {page_start}-{page_end}",
        fontsize=12,
        fontweight="bold",
    )
    fig.supxlabel(f"Sampled weight index ({SAMPLES_PER_BLOCK} weights per block)")
    fig.supylabel("|W_after - W_before|")
    plt.tight_layout(rect=(0.03, 0.03, 1, 0.96))
    plt.show()

peak_block = max(range(len(all_deltas)), key=lambda i: all_deltas[i].mean())  # block with the largest average movement
min_block = min(range(len(all_deltas)), key=lambda i: all_deltas[i].mean())  # block with the smallest average movement
print(
    f"Mean |delta W| by block - min: block {min_block} "
    f"({all_deltas[min_block].mean():.4e}),  "
    f"max: block {peak_block} ({all_deltas[peak_block].mean():.4e}).  "
    f"Each panel shows {SAMPLES_PER_BLOCK} sampled individual-weight deltas."
)

del ft_trace_model
gc.collect()  # free the reloaded model's memory now that its deltas are captured
print("Freed ft_trace_model from memory (checkpoint still on disk).")


### Visualizing Training Progress: Loss Curves

After training completes, inspect the real per-step loss recorded by the `Trainer`, not an idealized illustration. Textbook loss curves are smooth; a 10-step, batch-size-1 CPU demo is usually much noisier, and that is worth seeing honestly.

**What to look for:**

1. **Overall direction:** Compare the first and final logged losses without demanding monotonic progress.
2. **Batch noise:** Each point comes from one paragraph-sized batch, so example difficulty can dominate adjacent steps.
3. **Magnitude:** Lower training loss means a better fit to these batches, not proof of held-out quality.

With all ten steps logged, inspect the complete path rather than one endpoint. A production run would add held-out loss and stop from measured convergence rather than this fixed teaching budget.

> **PyTorch → Keras:** `trainer.state.log_history` — HuggingFace's `Trainer` records a running list of dicts (step number, loss, learning rate, etc.) logged every `logging_steps`; this cell filters that list down to just the `(step, loss)` pairs for plotting. **Keras/TF equivalent:** `history = model.fit(...)` / `history.history["loss"]` — Keras's `fit()` returns a `History` object whose `.history` dict holds per-*epoch* (not per-step, by default) metric lists; matching HuggingFace's per-step granularity in Keras needs a custom callback (e.g. overriding `on_train_batch_end`).

In [ ]:
# Visualize the REAL loss curve from the continued-pretraining run above (trainer_full),
# not a fabricated "typical" curve -- this is exactly what your training just did.
def extract_loss_history(trainer):

    # Pull (step, loss) pairs out of the Trainer's log history, skipping eval-only entries
    return [
        (entry["step"], entry["loss"])
        for entry in trainer.state.log_history
        if "loss" in entry
    ]


full_ft_history = extract_loss_history(trainer_full)
steps, losses = zip(*full_ft_history)  # split into two parallel sequences for plotting

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(
    steps,
    losses,
    marker="o",
    linewidth=2,
    markersize=7,
    color="green",
    label="Training loss",
)
ax.set_xlabel("Training Step")
ax.set_ylabel("Loss")
ax.set_title(
    f"Continued Pretraining (Full FT): real loss log ({losses[0]:.2f} \u2192 {losses[-1]:.2f})",
    fontsize=12,
    fontweight="bold",
)
ax.grid(alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

print(f"\n{'=' * 70}")
print("Reading this REAL loss curve (not an idealized one):")
print(f"{'=' * 70}")
print(f"  Logged steps: {list(steps)}")
print(f"  Logged losses: {[round(l, 3) for l in losses]}")
print(f"  First -> last: {losses[0]:.3f} -> {losses[-1]:.3f}")
print(f"{'=' * 70}")
print("What to look for:")
print(
    "  • A clean, monotonic plateau like a textbook figure is the exception, not the rule --"
)
print(
    "    especially at batch_size=1 with only a handful of steps, loss is dominated by"
)
print("    per-batch noise (which paragraph happened to be in this batch) more than by")
print("    the underlying trend.")
print(
    "  • If the trend is flat/noisy rather than decreasing: raise max_steps, increase the"
)
print(
    "    batch size, or train on more paragraphs so the trend has room to dominate the noise."
)
print(
    "  • Compare this to the loss curves for instruction tuning, partial freezing, and LoRA"
)
print(
    "    continued pretraining further down -- they were all recorded the same real way."
)
print(f"{'=' * 70}")


### Diagnose Continued-Pretraining Failures

| Symptom | Likely cause | First response |
| --- | --- | --- |
| General prompts become nonsense | Too many updates on a narrow corpus | Reduce steps and evaluate domain and general prompts together |
| Training perplexity approaches 1 while held-out perplexity stays high | Memorization | Add diverse text and deduplicate repeated passages |
| Most tokens are padding | `max_length` is much larger than typical paragraphs | Match block length to the observed token-length distribution |
| Training crashes around padding | Causal tokenizer has no pad token | Set `tokenizer.pad_token = tokenizer.eos_token` and mask padding labels with `-100` |

A quick health check needs three probes:

```python
generate(model, "Aria Voss checked the Meridian's Promise and")  # domain fit
generate(model, "The capital of France is")                      # retention
generate(model, "In the Under-Hold, the rebels gathered and")    # generalization
```

A failed general prompt suggests forgetting. A word-for-word held-out continuation suggests memorization. Neither is visible from training loss alone.

## Concept 2: Supervised Fine-Tuning - Teach the Request/Response Contract

Continued pretraining changed the training experience but the 10-step output probe remains inconclusive; the next observed gap is that a request such as `Answer in one sentence and stop` is still just more text to continue.

**Observed blocker:** domain fluency is not instruction compliance.

Riverside now supplies demonstrations with two roles:

- **Request:** `Continue this passage with one paragraph in the same style.`
- **Desired response:** the next manuscript paragraph.

The model must read both parts, but only the response is its work product. That creates the need for **prompt masking**: request tokens remain visible as context while their labels become `-100`, so the loss grades response tokens only.

Training on many request/response demonstrations is **supervised fine-tuning (SFT)**. The intuition comes before the implementation:

1. show the task contract;
2. show a response that satisfies it;
3. update the model toward the response, not toward reproducing the request.

This notebook derives a small private dataset from adjacent Riverside paragraphs. That is enough to demonstrate the pipeline and the practiced continuation instruction, but it is not evidence of broad instruction following. A production suite needs varied real editor requests, held-out cases, source-support checks, and explicit pass criteria.

**Checkpoint after training:** rerun the bounded-request probe. If the SFT adapter follows the requested shape, compare several valid responses to the same prompt. SFT can imitate a demonstrated answer, but it does not directly express why one acceptable answer is better than another.

> **Bridge to preference alignment:** once outputs are valid, production systems need preference alignment to distinguish concise, useful answers from technically correct but bloated ones. The next structural proxy only demonstrates those mechanics; it does not establish real editorial preference.

This teaching run uses LoRA to fit local hardware. Part 2 separates that parameter choice from the SFT objective.

> **PyTorch → Keras:** `from peft import LoraConfig, get_peft_model, TaskType` — imports HuggingFace's PEFT library, which wraps a PyTorch model's targeted `nn.Linear` layers with low-rank adapter matrices and freezes everything else; the actual wrapping happens a few cells down. **Keras/TF equivalent:** there is no first-party `peft` support for `TFPreTrainedModel`s — the common Keras/TF pattern for parameter-efficient tuning is manual layer freezing (`layer.trainable = False` on all but the last few layers) rather than LoRA adapters, since PEFT's LoRA implementation is PyTorch-only.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

# The baseline and full-FT models are no longer needed after their comparison and loss plot.
del trainer_full, full_ft_model, base_model, base_state
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Freed the baseline and full-FT models before LoRA SFT.")

# This section defines the instruction task and a function to build instruction-response pairs from novels.
# The same instruction guideline is applied to all novels to maintain consistency in the instruction-response pairs.
# In practice
INSTRUCTION_TASK = "Continue the fiction narrative in the same style."


def build_instruction_pairs(novels=None, max_chapters=None):
    if novels is None:
        novels = ["scifi", "fantasy", "mystery", "cyberpunk", "literary"]

    pairs = []
    for alias in novels:
        novel_dir = NOVELS.get(alias, "the-weight-of-distant-light")
        novel_path = CONTENT_DIR / novel_dir
        files = sorted(novel_path.glob("chapter-*.txt"))
        if max_chapters is not None:
            files = files[:max_chapters]
        for path in files:
            paragraphs = [
                paragraph.strip().replace("\n", " ")
                for paragraph in path.read_text(encoding="utf-8").split("\n\n")
                if len(paragraph.strip()) > 200
            ]
            for context, response in zip(paragraphs, paragraphs[1:]):
                instruction = f"{INSTRUCTION_TASK}\n\nContext:\n{context}"
                pairs.append({"instruction": instruction, "response": response})
    return pairs


### Where These Instruction Pairs Actually Come From

This notebook does **not** download or run Mistral, Phi-2, GPT-4o, or any separate pair-generation model. The runnable path constructs examples deterministically from Riverside's existing manuscripts:

```text
paragraph i     -> instruction context
paragraph i + 1 -> desired continuation
```

For each adjacent pair, `build_instruction_pairs()` creates one fixed request:

```text
Continue the fiction narrative in the same style.

Context:
<paragraph i>
```

The response is the next real paragraph. That is why the notebook can build 4,080 examples locally without another model or API call.

| Role | What this notebook uses | Downloaded or executed here? |
| --- | --- | --- |
| Pair construction | Python adjacency logic over Riverside paragraphs | Yes; no model required |
| Model being fine-tuned | `HuggingFaceTB/SmolLM2-360M-Instruct` from Hugging Face | Yes |
| Optional synthetic-pair generator | An instruction model such as `mistralai/Mistral-7B-Instruct-v0.3` | No |

This extraction approach is useful for teaching prompt masking, LoRA SFT, and checkpoint flow, but it has a narrow teaching signal: every example asks for continuation, and the target is copied from the following paragraph. It does **not** create diverse editing, summarization, question-answering, or style-control instructions.

A production data pipeline could load a separate Hugging Face-hosted instruction model, ask it for structured instruction-response JSON, then validate, deduplicate, provenance-tag, and review those generated examples. That would be a separate data-generation stage with its own runtime and quality evaluation; it is not implied by the code in this notebook.

### Tokenizing With the Prompt-Mask Pattern

Token-aware budgeting keeps at most 64 native prompt tokens and at most 32 native assistant-suffix tokens inside 96 positions. Prompt and padding labels are `-100`, and exactly one assistant EOS token is supervised.


> **PyTorch → Keras:** `tokenize_instruction()` — builds `labels` as a copy of the tokenized `input_ids`, then overwrites *both* the prompt-token positions and the padding positions with `-100`, so a cross-entropy loss with `ignore_index=-100` (used earlier in the notebook) only ever grades the completion tokens. **Keras/TF equivalent:** the same masking logic — a Keras/TF version would build an analogous `labels` array with `-100` (or `0` plus a matching `sample_weight` mask, since TF's `SparseCategoricalCrossentropy` has no built-in `ignore_index`) at prompt+padding positions; the tokenization itself is identical since `AutoTokenizer` is framework-agnostic.

In [ ]:
_SFT_CONTEXT_MARKER = "\n\nContext:\n"


def _sft_token_ids(text):
    return tokenizer(text, add_special_tokens=False)["input_ids"]


def _sft_prefix_encoding(text):
    if tokenizer.is_fast:
        return tokenizer(text, add_special_tokens=False, return_offsets_mapping=True)
    return tokenizer(text, add_special_tokens=False)


def _sft_token_prefix_text(text, encoding, token_count):
    if token_count == 0:
        return ""
    offsets = encoding.get("offset_mapping")
    if offsets is not None:
        return text[: offsets[token_count - 1][1]].rstrip()
    return tokenizer.decode(
        encoding["input_ids"][:token_count],
        skip_special_tokens=False,
        clean_up_tokenization_spaces=False,
    ).rstrip()


def _sft_largest_fitting_prefix(text, max_source_tokens, build_candidate):
    encoding = _sft_prefix_encoding(text)
    low = 0
    high = min(len(encoding["input_ids"]), max_source_tokens)
    best = None

    while low <= high:
        token_count = (low + high) // 2
        prefix = _sft_token_prefix_text(text, encoding, token_count)
        candidate = build_candidate(prefix)
        if candidate is None:
            high = token_count - 1
        else:
            best = candidate
            low = token_count + 1

    return best


def _sft_bounded_prompt(instruction, prompt_max_length):
    task, marker, context = instruction.strip().partition(_SFT_CONTEXT_MARKER)
    if not marker:
        raise ValueError("Instruction is missing the expected Context section")

    context = context.strip()
    instruction_prefix = task + marker

    def build_candidate(bounded_context):
        bounded_instruction = instruction_prefix + bounded_context
        prompt_text = render_instruction(bounded_instruction)
        prompt_ids = _sft_token_ids(prompt_text)
        if len(prompt_ids) <= prompt_max_length:
            return bounded_instruction, prompt_text, prompt_ids
        return None

    result = _sft_largest_fitting_prefix(
        context, prompt_max_length, build_candidate
    )
    if result is None:
        raise ValueError("The fixed instruction template exceeds the prompt budget")
    return result


def _sft_bounded_assistant_suffix(
    bounded_instruction,
    prompt_text,
    prompt_ids,
    response,
    assistant_max_length,
):
    if tokenizer.eos_token is None or tokenizer.eos_token_id is None:
        raise ValueError("The tokenizer must define an EOS token")

    def build_candidate(bounded_response):
        full_text = render_instruction(bounded_instruction, bounded_response)
        assert full_text.startswith(prompt_text), (
            "Native chat template did not preserve the exact prompt text prefix"
        )

        assistant_text = full_text[len(prompt_text) :]
        eos_offset = assistant_text.rfind(tokenizer.eos_token)
        if eos_offset < 0:
            raise ValueError("Native assistant rendering did not contain EOS")

        through_eos_text = prompt_text + assistant_text[
            : eos_offset + len(tokenizer.eos_token)
        ]
        through_eos_ids = _sft_token_ids(through_eos_text)
        if through_eos_ids[: len(prompt_ids)] != prompt_ids:
            return None

        assistant_ids = through_eos_ids[len(prompt_ids) :]
        if (
            len(assistant_ids) <= assistant_max_length
            and assistant_ids
            and assistant_ids[-1] == tokenizer.eos_token_id
            and assistant_ids.count(tokenizer.eos_token_id) == 1
        ):
            return assistant_ids
        return None

    result = _sft_largest_fitting_prefix(
        response.strip(), assistant_max_length, build_candidate
    )
    if result is None:
        raise ValueError("No stable native assistant suffix fits the assistant budget")
    return result


def tokenize_instruction(
    example,
    max_length=96,
    prompt_max_length=64,
    assistant_max_length=32,
):
    if prompt_max_length + assistant_max_length != max_length:
        raise ValueError("Prompt and assistant budgets must sum to max_length")
    if tokenizer.pad_token_id is None:
        raise ValueError("The tokenizer must define a padding token")

    bounded_instruction, prompt_text, prompt_ids = _sft_bounded_prompt(
        example["instruction"], prompt_max_length
    )
    assistant_ids = _sft_bounded_assistant_suffix(
        bounded_instruction,
        prompt_text,
        prompt_ids,
        example["response"],
        assistant_max_length,
    )

    active_ids = prompt_ids + assistant_ids
    padding_length = max_length - len(active_ids)
    if padding_length < 0:
        raise AssertionError("Bounded prompt and assistant suffix exceed max_length")

    input_ids = active_ids + [tokenizer.pad_token_id] * padding_length
    attention_mask = [1] * len(active_ids) + [0] * padding_length
    labels = [-100] * len(prompt_ids) + assistant_ids + [-100] * padding_length

    assert assistant_ids[-1] == tokenizer.eos_token_id
    assert assistant_ids.count(tokenizer.eos_token_id) == 1
    assert len(input_ids) == len(attention_mask) == len(labels) == max_length
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }


### Building the Instruction Dataset

Now actually build the pairs from 5 genres and tokenize every one of them with the function above.


In [ ]:
instruction_pairs = build_instruction_pairs(
    novels=["scifi", "fantasy", "mystery", "cyberpunk", "literary"],
    max_chapters=None,  # None = all available chapters
)
print(f"Built {len(instruction_pairs)} instruction pairs from 5 novels")

instruction_dataset = Dataset.from_list(instruction_pairs)  # wrap the prompt/completion pairs in a HF Dataset
instruction_tokenized = instruction_dataset.map(
    tokenize_instruction, remove_columns=["instruction", "response"]
)  # apply the prompt-masking tokenizer across every pair

for row in instruction_tokenized:
    input_ids = row["input_ids"]
    attention_mask = row["attention_mask"]
    labels = row["labels"]

    assert len(input_ids) == len(attention_mask) == len(labels) == 96
    prompt_boundary = next(
        index for index, label in enumerate(labels) if label != -100
    )
    active_length = sum(attention_mask)
    active_response_labels = labels[prompt_boundary:active_length]

    assert all(
        label == -100 or mask == 1
        for label, mask in zip(labels, attention_mask)
    )
    assert all(label == -100 for label in labels[:prompt_boundary])
    assert all(label != -100 for label in active_response_labels)
    assert all(label == -100 for label in labels[active_length:])
    assert prompt_boundary <= 64
    assert len(active_response_labels) <= 32
    assert active_response_labels.count(tokenizer.eos_token_id) == 1
    assert active_response_labels[-1] == tokenizer.eos_token_id

print(
    f"PASS: all {len(instruction_tokenized)} SFT rows preserve the "
    "64/32 prompt-response boundary."
)


### A Quick LoRA Preview for This SFT Run

SFT defines **what behavior is taught**. LoRA only changes **where the update is stored**: `get_peft_model()` freezes the base and adds small trainable correction matrices to selected attention projections.

That is enough detail for this chapter. The next code cell uses LoRA so the SFT run fits local hardware; Part 2 derives the low-rank path, measures its parameter budget, and inspects the real matrices.

> **PyTorch → Keras:** `LoraConfig(...)` and `get_peft_model(...)` target the four SmolLM2 attention
projections (`q_proj`, `k_proj`, `v_proj`, `o_proj`) and report the resulting trainable fraction.
PEFT's wrapping is PyTorch-specific; a Keras implementation needs a compatible low-rank layer wrapper or
manual custom layers rather than coarse whole-layer freezing.


In [ ]:
# LoRA hyperparameters: rank-8 adapters on all four attention projections.
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
)

instruct_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
instruct_lora_model = get_peft_model(instruct_base, lora_config)
instruct_lora_model.print_trainable_parameters()


### Training and Saving the Adapter

Same `Trainer` pattern as continued pretraining, just with the LoRA-wrapped model, the prompt-masked
dataset, and a higher learning rate (`2e-4` vs. `5e-5`) -- LoRA needs a higher LR since it's only
updating a tiny slice of parameters.


> **PyTorch → Keras:** `Trainer(model=instruct_lora_model, ...)` / `trainer_instruct.train()` / `instruct_lora_model.save_pretrained(...)` — the same HuggingFace `Trainer` pattern as the earlier full fine-tuning run, just pointed at the LoRA-wrapped model and the prompt-masked instruction dataset, with a higher learning rate since only the small adapter matrices are being updated. **Keras/TF equivalent:** `model.fit(dataset, epochs=...)` — as with the earlier full fine-tuning cell, a Keras/TF version would call `.fit()` on the (layer-frozen) model instead of `Trainer.train()`; `save_pretrained()` again has an identically-named counterpart on `TFPreTrainedModel`.

In [ ]:
# Configure a short LoRA training run with a higher LR (only the adapter matrices are trainable)
training_args_instruct = TrainingArguments(
    output_dir="./checkpoints/instruction-lora",
    per_device_train_batch_size=1,
    max_steps=DEMO_TRAIN_STEPS,  # short demonstration run; tune this from measured convergence
    logging_steps=1,
    save_strategy="no",
    learning_rate=2e-4,
    report_to="none",
)

# Wrap the LoRA-wrapped model, config, and tokenized instruction dataset in a Trainer and run it
trainer_instruct = Trainer(
    model=instruct_lora_model,
    args=training_args_instruct,
    train_dataset=instruction_tokenized,
)
trainer_instruct.train()
instruct_lora_model.save_pretrained("./checkpoints/instruction-lora")  # persist the adapter weights only
print("Saved instruction-tuned LoRA adapter.")


### Instruction Tuning, Recapped

The preceding cells implement one SFT pipeline:

1. `build_instruction_pairs()` turns adjacent paragraphs into an editor request and desired continuation.
2. `tokenize_instruction()` keeps the request visible but masks its labels with `-100`, so loss grades only the assistant response.
3. A LoRA wrapper keeps the base frozen and stores this teaching run's update in a small adapter.
4. `Trainer.train()` batches the examples and updates only the adapter parameters.

```text
Continued pretraining: [labels for text .....................] [pad: -100]
Instruction tuning:   [prompt: -100 ........] [completion labels] [pad: -100]
```

The conceptual change is the supervision boundary: Riverside now grades what the assistant should return for a request instead of every input token.

### The Instruction-Tuning Mask Layout, For Real

Contrast this with the continued-pretraining mask layout earlier in the notebook: there, only **padding** was masked, and every real token was active. Here, the **bounded prompt context is masked**, while the **bounded assistant suffix is supervised**. The model is penalized only for the assistant suffix, including exactly one EOS token, never for reproducing the prompt it was given. The cell below takes one real `(prompt, completion)` pair from `instruction_pairs`, runs it through the real `tokenize_instruction()` used for training, and colors every token position by what the label mask actually does with it.


In [ ]:
# Real mask layout for instruction tuning: prompt masked (-100), completion active, padding masked
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

example_pair = instruction_pairs[0]
encoded_example = tokenize_instruction(example_pair)
example_labels = np.array(encoded_example["labels"])
example_attention = np.array(encoded_example["attention_mask"])

# Classify every position: 0 = masked prompt, 1 = active completion, 2 = masked padding
region = np.zeros(len(example_labels), dtype=int)
region[example_attention == 0] = 2  # padding
region[(example_attention == 1) & (example_labels != -100)] = 1  # completion (active)

# everything else (attention==1 & labels==-100) is the masked prompt, stays 0

prompt_masked = int(np.sum(region == 0))
completion_active = int(np.sum(region == 1))
padding_masked = int(np.sum(region == 2))

# Visualize the mask layout as a single color-coded strip (gray=prompt, green=completion, white=padding)
fig, ax = plt.subplots(figsize=(14, 2.2))
cmap = ListedColormap(["lightgray", "mediumseagreen", "white"])
ax.imshow(
    region.reshape(1, -1),
    cmap=cmap,
    aspect="auto",
    vmin=0,
    vmax=2,
    extent=[0, len(region), 0, 1],
)
ax.set_yticks([])
ax.set_xlabel("Token position")
ax.set_title(
    f"{prompt_masked} prompt tokens masked + {completion_active} completion tokens active "
    f"+ {padding_masked} padding tokens masked",
    fontsize=10,
    fontweight="bold",
)

# Legend entries matching each color band in the strip above
legend_handles = [
    Patch(
        facecolor="lightgray", edgecolor="black", label="Prompt (masked, labels=-100)"
    ),
    Patch(
        facecolor="mediumseagreen",
        edgecolor="black",
        label="Completion (active, real labels)",
    ),
    Patch(facecolor="white", edgecolor="black", label="Padding (masked, labels=-100)"),
]
ax.legend(
    handles=legend_handles,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.55),
    ncol=3,
    fontsize=9,
)

plt.tight_layout()
plt.show()

print(f"Prompt text:     {render_instruction(example_pair['instruction'])[:80]!r}...")
print(f"Completion text: {example_pair['response'][:80]!r}...")
print(
    f"\nMasked (prompt): {prompt_masked} tokens | Active (completion): {completion_active} tokens "
    f"| Masked (padding): {padding_masked} tokens"
)
print(
    "Compare to continued pretraining: there, every real token was active. Here, the prompt is "
    "masked too, so the model only ever gets gradient signal from the completion."
)

### Diagnose Instruction-Tuning Failures

| Symptom | Likely cause | First response |
| --- | --- | --- |
| Model echoes or predicts the request | Prompt tokens were included in the loss | Set prompt and padding labels to `-100` |
| Works only with one exact prefix | Template overfitting | Diversify training instructions or enforce the same production template |
| Responses stop much too early or run too long | Completion-length distribution mismatches the task | Build examples with production-like response lengths |
| LoRA barely learns | Learning rate is too low for the small trainable state | Tune the adapter learning rate; do not inherit full-FT defaults blindly |

Typical starting ranges are `5e-5` to `1e-4` for full fine-tuning, `1e-4` to `2e-4` for partial freezing, and `2e-4` to `5e-4` for LoRA. They are search ranges, not guarantees.

The next code cell compares the trained model with and without the expected prefix, then tests a novel request. That distinguishes template dependence from broader failure to generalize.

In [ ]:
# Quick health check after instruction tuning

instruction_prompt = render_instruction(
    "Continue the fiction narrative in the same style.\n\nContext:\nAria checked the Meridian and"
)
print("=== Test 1: Trained instruction ===")
print(f"  Input : {instruction_prompt!r}")
print(f"  Output: {generate(instruct_lora_model, instruction_prompt)}")
print()

raw_prompt = "Aria checked the Meridian and"
print("=== Test 2: Plain text (checks template dependence) ===")
print(f"  Input : {raw_prompt!r}")
print(f"  Output: {generate(instruct_lora_model, raw_prompt)}")
print()

novel_prompt = render_instruction(
    "Continue the fiction narrative in the same style.\n\nContext:\nIn the Upper decks, Marcus"
)
print("=== Test 3: Novel instruction (generalization check) ===")
print(f"  Input : {novel_prompt!r}")
print(f"  Output: {generate(instruct_lora_model, novel_prompt)}")


## Concept 3: Preference Alignment - When Imitation Is Not Enough

SFT fixed the interaction contract. It can still leave Riverside with two valid answers where one is clearly more useful to an editor.

For the same request:

- **Production `chosen` semantics:** concise, advances the scene, and preserves supplied facts.
- **Production `rejected` semantics:** grammatical, but repetitive or less useful.

SFT teaches by imitation: “produce answers like this example.” It does not directly teach: “when these two answers compete, prefer this one.” Preference alignment adds that missing comparison.

### Why not only suppress the rejected response?

Telling the model only what to avoid leaves too many escape routes. It might become terse, shift probability to another bad answer, exploit response length, or forget useful SFT behavior. A preference pair supplies a direction: move toward the chosen behavior relative to the rejected behavior.

### The preference-data contract

| Field | Meaning |
| --- | --- |
| `prompt` | One shared editor request |
| `chosen` | The response the annotator prefers |
| `rejected` | A less-preferred response to that same request |

In production, chosen/rejected qualities are the target semantics and should come from real comparisons such as blinded editor judgments, accept-versus-rewrite behavior, or a calibrated judge. This notebook uses a structural proxy instead: equal-budget bounded prefixes from an adjacent paragraph (`chosen`) and an unrelated paragraph (`rejected`). That exposes the mechanics, but it does not establish real editor validity.

### The frozen SFT model is the “before” photo

DPO starts with two copies of the accepted SFT model:

1. **Live policy:** the copy training may change.
2. **Frozen reference:** the untouched SFT snapshot.

Think of the reference as a before photo. For each candidate response, DPO asks: **did the live policy move toward or away from this response compared with where SFT started?**

The useful comparison is not whether the chosen response has a large score by itself. It is whether the chosen response gained more ground than the rejected response.

**Predict:** At the first step, before the live copy changes, which response has gained more ground?

**Expected:** neither. The live and frozen copies are identical, so the comparison begins as a tie. Training should break that tie in the chosen response's favor.


### From Reward-Model RLHF to DPO: Two Paths from the Same Feedback

Both PPO-based RLHF and DPO start with the same evidence:

> A person compares two answers to the same request and identifies the better one.

They also share one safety principle: improve preference without casually discarding useful behavior learned during SFT. Their difference is how they turn comparisons into policy updates.

```mermaid
flowchart TD
    H["Human comparisons"] --> RM["Reward model"]
    RM --> R["Reward-scored<br/>fresh rollouts"]
    R --> PPO["PPO update"]
    REF["Frozen SFT reference"] --> PPO
    PPO --> PP["Updated policy"]
    H --> DPO["DPO direct path"]
    REF --> M["Compare chosen vs rejected<br/>movement"]
    DPO --> M
    M --> DP["Updated policy"]
```

#### The reward-model RLHF path

A reward model first learns to predict which answer people prefer. The policy then generates fresh answers, the reward model scores them, and PPO updates the policy. A frozen SFT model discourages the live policy from drifting too far merely to exploit the reward model.

That route can explore behavior absent from the original comparison set, but it brings several moving parts:

- a policy, frozen reference, reward model, and usually a value model;
- generation inside the training loop;
- interacting reward, clipping, value, and drift settings;
- failure modes such as reward hacking and reward-model distribution shift.

#### The shared anchor, used two ways

The frozen SFT model serves the same broad purpose in both paths: preserve a useful starting behavior while preference learning changes the policy.

- **PPO-based RLHF:** explicitly penalizes the policy for moving too far from SFT.
- **DPO:** includes the frozen SFT scores inside each chosen-versus-rejected comparison.

So the intuition transfers, but the mechanics differ. PPO uses the reference as a drift penalty; DPO uses it as the baseline that makes relative movement meaningful.

#### The DPO direct path

DPO is not “PPO, only better.” It chooses a different trade-off: when curated offline pairs already capture the desired behavior, optimize those pairs directly instead of training a reward model and generating fresh rollouts. The gain is simpler offline training; the cost is losing PPO's online exploration.

Use two scoreboards:

1. The **frozen SFT scoreboard** records where both responses started.
2. The **live policy scoreboard** records where training moved them.

For each response, compare the live score with its frozen starting score. Because chosen and rejected answer the same prompt, comparing their movements focuses the signal on **which response gained relative ground**, rather than on a prompt-wide score shared by both.

> **Preference edge = chosen movement from SFT minus rejected movement from SFT.**

- **Positive edge:** the chosen response gained more ground.
- **Zero edge:** training has not separated the pair.
- **Negative edge:** the rejected response gained more ground.

The trainer converts this edge into a smooth loss. There is no hard pass/fail boundary: wrong-direction pairs receive stronger correction, and already-correct pairs still receive smaller updates. TRL handles the exact probability algebra; the edge is the useful mental model.

A positive edge can happen in three valid ways: raise the chosen response, lower the rejected response, or lower both while lowering the rejected response more. DPO cares about the relative direction, not one response's absolute score.

### DPO and PPO-RLHF Solve Different Operational Problems

| | DPO | PPO-based RLHF |
| --- | --- | --- |
| Training evidence | Fixed offline preference pairs | Fresh generated behavior scored during training |
| Extra learned models | No reward or value model | Reward model and usually a value model |
| Main strength | Simpler offline optimization | Online exploration beyond the pair dataset |
| Main risk | Limited by pair quality and coverage | Reward hacking, instability, and infrastructure cost |
| Prefer when | Curated pairs capture the target behavior | Environment feedback or online exploration is essential |

Riverside uses DPO here because the teaching data is already a fixed set of chosen/rejected pairs. The next cell makes the preference edge concrete without deriving the loss.


In [ ]:
# Step 1: build intuition with a movement scoreboard rather than a loss derivation.
import gc
import torch
import torch.nn.functional as F

# Release completed or stale model objects before loading two fresh DPO copies.
model_names_to_release = (
    "trainer_instruct",
    "instruct_lora_model",
    "instruct_base",
    "dpo_trainer",
    "dpo_policy_model",
    "dpo_policy_base",
    "dpo_reference_model",
    "dpo_reference_base",
)
for model_name in model_names_to_release:
    if model_name in globals():
        del globals()[model_name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Freed completed model state before loading the DPO policy and reference.")


def preference_edge(chosen_movement, rejected_movement):
    """Positive means the chosen response gained more ground from SFT."""
    return chosen_movement - rejected_movement


def edge_reading(edge):
    if edge > 0:
        return "chosen gained more"
    if edge < 0:
        return "rejected gained more"
    return "tie"


movement_scenarios = [
    ("identical SFT copies", 0.0, 0.0),
    ("wrong direction", -0.2, 0.3),
    ("raise chosen only", 0.4, 0.0),
    ("lower rejected only", 0.0, -0.4),
    ("lower both; reject more", -0.1, -0.5),
]

print(f"{'Scenario':27s} {'Chosen move':>12s} {'Rejected move':>14s} {'Edge':>7s}  Reading")
for label, chosen_movement, rejected_movement in movement_scenarios:
    edge = preference_edge(chosen_movement, rejected_movement)
    print(
        f"{label:27s} {chosen_movement:+12.1f} {rejected_movement:+14.1f} "
        f"{edge:+7.1f}  {edge_reading(edge)}"
    )

assert preference_edge(0.0, 0.0) == 0.0
assert preference_edge(-0.2, 0.3) < 0.0
assert preference_edge(0.4, 0.0) > 0.0
assert preference_edge(0.0, -0.4) > 0.0
assert preference_edge(-0.1, -0.5) > 0.0
print("PASS: different movements can create the same chosen-over-rejected edge.")


In [ ]:
# Step 2: build offline preference triples with one shared prompt and two responses.
# Teaching proxy: adjacent text is always chosen and unrelated text is always rejected.
# Real human preference data is subtler, noisier, and sometimes contradictory.
DPO_MAX_LENGTH = 128
DPO_MAX_PROMPT_TOKENS = 64
DPO_MAX_RESPONSE_TOKENS = DPO_MAX_LENGTH - DPO_MAX_PROMPT_TOKENS - 1


def _token_count(text):
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])


def _bounded_prompt(context):
    """Keep as much context as fits beside the chat-template overhead."""
    context_ids = tokenizer(context, add_special_tokens=False)["input_ids"]
    context_ids = context_ids[:DPO_MAX_PROMPT_TOKENS]
    while context_ids:
        bounded_context = tokenizer.decode(context_ids, skip_special_tokens=True).strip()
        instruction = f"{INSTRUCTION_TASK}\n\nContext:\n{bounded_context}"
        prompt = render_instruction(instruction)
        if _token_count(prompt) <= DPO_MAX_PROMPT_TOKENS:
            return instruction, prompt
        context_ids = context_ids[:-1]
    raise ValueError("Chat-template overhead leaves no room for DPO context tokens")


def _bounded_response(instruction, prompt, response):
    """Keep a response prefix that remains fully visible under DPO_MAX_LENGTH."""
    response_ids = tokenizer(response, add_special_tokens=False)["input_ids"]
    response_ids = response_ids[:DPO_MAX_RESPONSE_TOKENS]
    while response_ids:
        bounded_text = tokenizer.decode(response_ids, skip_special_tokens=True).strip()
        full_text = render_instruction(instruction, bounded_text)
        if not full_text.startswith(prompt):
            raise ValueError("Instruction template did not preserve the DPO prompt prefix")
        response_suffix = full_text[len(prompt) :]
        eos_index = response_suffix.find(tokenizer.eos_token)
        if eos_index == -1:
            raise ValueError("Rendered response does not contain the tokenizer EOS marker")
        response_suffix = response_suffix[: eos_index + len(tokenizer.eos_token)]
        if _token_count(response_suffix) <= DPO_MAX_RESPONSE_TOKENS:
            return response_suffix
        response_ids = response_ids[:-1]
    raise ValueError("No response tokens fit inside the DPO sequence budget")


def build_preference_pairs(novels=None, max_chapters=4, max_pairs=30):
    if novels is None:
        novels = ["scifi", "fantasy", "mystery", "horror", "literary"]

    chapter_files = []
    for alias in novels:
        novel_dir = NOVELS.get(alias, "the-weight-of-distant-light")
        novel_path = CONTENT_DIR / novel_dir
        files = sorted(novel_path.glob("chapter-*.txt"))
        chapter_files.extend(files if max_chapters is None else files[:max_chapters])

    all_paragraphs = []
    for path in chapter_files:
        paragraphs = [
            paragraph.strip().replace("\n", " ")
            for paragraph in path.read_text(encoding="utf-8").split("\n\n")
            if len(paragraph.strip()) > 200
        ]
        if len(paragraphs) >= 2:
            all_paragraphs.append(paragraphs)

    pairs = []
    for chapter_index, paragraphs in enumerate(all_paragraphs):
        other_chapter = all_paragraphs[(chapter_index + 1) % len(all_paragraphs)]
        for paragraph_index in range(len(paragraphs) - 1):
            instruction, prompt = _bounded_prompt(paragraphs[paragraph_index])
            chosen = _bounded_response(
                instruction, prompt, paragraphs[paragraph_index + 1]
            )
            rejected = _bounded_response(
                instruction,
                prompt,
                other_chapter[paragraph_index % len(other_chapter)],
            )
            pairs.append({"prompt": prompt, "chosen": chosen, "rejected": rejected})

    return pairs if max_pairs is None else pairs[:max_pairs]


preference_pairs = build_preference_pairs()

# TRL appends EOS when a response suffix does not literally end with the EOS string.
for pair in preference_pairs:
    prompt_tokens = _token_count(pair["prompt"])
    for response_key in ("chosen", "rejected"):
        response = pair[response_key]
        training_response = (
            response if response.endswith(tokenizer.eos_token) else response + tokenizer.eos_token
        )
        response_tokens = _token_count(training_response)
        if prompt_tokens + response_tokens > DPO_MAX_LENGTH:
            raise ValueError(f"{response_key} exceeds the DPO sequence budget")
        combined_ids = tokenizer(
            pair["prompt"] + training_response, add_special_tokens=False
        )["input_ids"]
        prompt_ids = tokenizer(pair["prompt"], add_special_tokens=False)["input_ids"]
        if combined_ids[: len(prompt_ids)] != prompt_ids:
            raise ValueError(f"Unstable tokenizer boundary for {response_key} response")

example_preference = preference_pairs[0]
print(
    f"Built {len(preference_pairs)} bounded preference pairs | "
    f"columns: {list(example_preference)}"
)
print("\n=== One offline comparison ===")
print(f"Shared prompt ({_token_count(example_preference['prompt'])} tokens):")
print(f"  {example_preference['prompt'][-220:]!r}")
print(f"Chosen response ({_token_count(example_preference['chosen'])} tokens):")
print(f"  {example_preference['chosen'][:220]!r}")
print(f"Rejected response ({_token_count(example_preference['rejected'])} tokens):")
print(f"  {example_preference['rejected'][:220]!r}")
print(
    f"\nEvery prompt/response/EOS sequence fits the {DPO_MAX_LENGTH}-token trainer budget."
)
print("The label is comparative: chosen is preferred to rejected for this same prompt.")


### From the Toy Scoreboard to Real Responses

The toy scoreboard treated each response as one item. A language model scores a response token by token, so the helper below adds the response-token scores into one overall response score. Prompt tokens provide context but are not counted as part of the response.

The same scoreboard is then filled four times behind the scenes:

- live policy on the chosen response;
- live policy on the rejected response;
- frozen SFT reference on the chosen response;
- frozen SFT reference on the rejected response.

Those internal scores reduce to the two values that matter for intuition: **chosen movement** and **rejected movement**. Their difference is the preference edge.

Both suffixes in this trace are bounded to equal 63-token budgets, with one EOS token inside the 128-token total sequence budget. That removes chosen/rejected length imbalance from this mechanism trace. In production, longer responses can still create shortcuts, so preference data should avoid systematic length differences and evaluation needs quality slices by response length.

The helper exposes the movement scoreboard before and after `DPOTrainer` runs. TRL handles the exact loss calculation during optimization.

> **PyTorch → Keras:** TRL's `DPOTrainer` is PyTorch-only. A Keras implementation would score both responses with live and frozen models, compare their movement from SFT, and optimize the resulting pairwise loss in a custom `train_step`.


In [ ]:
# Step 3: score the same bounded responses that DPOTrainer optimizes.
def response_sequence_logprob(model, prompt, response):
    """Return one response score using the trainer's EOS and token-budget contract."""
    training_response = (
        response if response.endswith(tokenizer.eos_token) else response + tokenizer.eos_token
    )
    prompt_ids = tokenizer(
        prompt, add_special_tokens=False, return_tensors="pt"
    ).input_ids
    full_ids = tokenizer(
        prompt + training_response, add_special_tokens=False, return_tensors="pt"
    ).input_ids

    prompt_length = prompt_ids.shape[1]
    if not torch.equal(full_ids[:, :prompt_length], prompt_ids):
        raise ValueError("Prompt tokens are not a stable prefix of prompt + response")
    if full_ids.shape[1] > DPO_MAX_LENGTH:
        raise ValueError("Diagnostic sequence exceeds the DPO trainer's token budget")

    model_device = next(model.parameters()).device
    full_ids = full_ids.to(model_device)
    model.eval()
    with torch.no_grad():
        logits = model(input_ids=full_ids).logits[:, :-1, :].float()
        token_logps = F.log_softmax(logits, dim=-1).gather(
            2, full_ids[:, 1:].unsqueeze(-1)
        ).squeeze(-1)

    # A target token at position i is scored by logits from position i - 1.
    response_logps = token_logps[:, prompt_length - 1 :]
    return {
        "sum": response_logps.sum().item(),
        "tokens": response_logps.shape[1],
    }


def measure_preference_snapshot(policy, reference, pair, beta):
    """Return the chosen/rejected movement edge and its training loss."""
    policy_chosen = response_sequence_logprob(policy, pair["prompt"], pair["chosen"])
    policy_rejected = response_sequence_logprob(policy, pair["prompt"], pair["rejected"])
    reference_chosen = response_sequence_logprob(
        reference, pair["prompt"], pair["chosen"]
    )
    reference_rejected = response_sequence_logprob(
        reference, pair["prompt"], pair["rejected"]
    )

    chosen_movement = policy_chosen["sum"] - reference_chosen["sum"]
    rejected_movement = policy_rejected["sum"] - reference_rejected["sum"]
    edge = chosen_movement - rejected_movement
    loss = F.softplus(torch.tensor(-beta * edge)).item()

    return {
        "chosen_tokens": policy_chosen["tokens"],
        "rejected_tokens": policy_rejected["tokens"],
        "chosen_movement": chosen_movement,
        "rejected_movement": rejected_movement,
        "preference_edge": edge,
        "loss": loss,
    }


def print_preference_snapshot(label, snapshot):
    print(f"\n=== {label} ===")
    print(
        f"Response lengths: chosen={snapshot['chosen_tokens']} tokens | "
        f"rejected={snapshot['rejected_tokens']} tokens"
    )
    print(f"Chosen movement from SFT:    {snapshot['chosen_movement']:+.3f}")
    print(f"Rejected movement from SFT:  {snapshot['rejected_movement']:+.3f}")
    print(f"Preference edge:             {snapshot['preference_edge']:+.3f}")
    print(f"Training loss:               {snapshot['loss']:.3f} (lower is better)")


print("Sequence scorer ready. The next cell compares the same pair before and after DPO.")


In [ ]:
# Step 4: compare the same movement scoreboard before and after DPO.
import gc

from peft import PeftModel
from trl import DPOConfig, DPOTrainer

DPO_BETA = 0.1
DPO_TRACKED_PAIR = preference_pairs[0]

# Policy and reference start from separate, identical SFT snapshots.
dpo_policy_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
dpo_reference_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
dpo_policy_model = PeftModel.from_pretrained(
    dpo_policy_base,
    "./checkpoints/instruction-lora",
    is_trainable=True,
).to(device)
dpo_reference_model = PeftModel.from_pretrained(
    dpo_reference_base,
    "./checkpoints/instruction-lora",
    is_trainable=False,
).to(device)
dpo_reference_model.eval()
for parameter in dpo_reference_model.parameters():
    parameter.requires_grad_(False)

print("Expected before DPO: chosen and rejected have moved equally, so the edge is zero.")
before_dpo = measure_preference_snapshot(
    dpo_policy_model,
    dpo_reference_model,
    DPO_TRACKED_PAIR,
    DPO_BETA,
)
print_preference_snapshot("Before DPO: live policy matches frozen SFT", before_dpo)
assert abs(before_dpo["preference_edge"]) < 1e-4

# Repeat the tracked bounded pair so this short run directly tests the mechanism it reports.
dpo_dataset = Dataset.from_list([DPO_TRACKED_PAIR] * DEMO_DPO_STEPS)
dpo_args = DPOConfig(
    output_dir="./checkpoints/preference-dpo",
    per_device_train_batch_size=1,
    max_length=DPO_MAX_LENGTH,
    max_steps=DEMO_DPO_STEPS,
    learning_rate=5e-5,
    beta=DPO_BETA,
    logging_steps=1,
    save_strategy="no",
    bf16=False,
    report_to="none",
)

dpo_trainer = DPOTrainer(
    model=dpo_policy_model,
    ref_model=dpo_reference_model,
    args=dpo_args,
    train_dataset=dpo_dataset,
    processing_class=tokenizer,
)
dpo_trainer.train()

after_dpo = measure_preference_snapshot(
    dpo_policy_model,
    dpo_reference_model,
    DPO_TRACKED_PAIR,
    DPO_BETA,
)
print_preference_snapshot("After DPO: same tracked training pair", after_dpo)
assert after_dpo["preference_edge"] > 0, (
    "The tracked pair did not move in the chosen-over-rejected direction."
)
print(
    "\nObserved change on this pair:\n"
    f"  preference edge: {before_dpo['preference_edge']:+.3f} -> "
    f"{after_dpo['preference_edge']:+.3f}\n"
    f"  training loss:  {before_dpo['loss']:.3f} -> {after_dpo['loss']:.3f}"
)
print(
    "This is a mechanism trace on a training pair, not held-out evidence of editor preference."
)

dpo_policy_model.save_pretrained("./checkpoints/preference-dpo")
tokenizer.save_pretrained("./checkpoints/preference-dpo")

# Keep downstream qualitative checks pointed at the aligned policy; release only the frozen copy.
instruct_lora_model = dpo_policy_model
del dpo_trainer, dpo_reference_model, dpo_reference_base
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Saved DPO-aligned adapter from a fresh SFT checkpoint.")


In [ ]:
# Quick health check after DPO


def _dpo_test(label, instruction):
    prompt = render_instruction(instruction)
    print(f"=== {label} ===")
    print(f"  Input : {prompt!r}")
    print(f"  Output: {generate(instruct_lora_model, prompt)}")
    print()


_dpo_test("Test 1: Preferred style", "Who is Aria Voss?")
_dpo_test(
    "Test 2: Still coherent on domain tasks",
    "Continue the fiction narrative: Aria checked the panel and",
)

print("=== Test 3: Same-prompt diversity smoke check ===")
instruction = "Describe the Meridian."
for attempt in range(1, 4):
    prompt = render_instruction(instruction)
    print(f"  Attempt {attempt}")
    print(f"  Output: {generate(instruct_lora_model, prompt)}")


### Reading the DPO Trace Honestly

Read the output as one scoreboard:

1. The frozen SFT model records the starting point.
2. The live policy scores the same chosen and rejected responses.
3. Each live score is compared with its own starting score.
4. Chosen movement minus rejected movement gives the preference edge.
5. Training tries to make that edge positive while keeping the SFT model as its anchor.

At initialization, both model copies are identical, so the edge is zero. After a useful update, the chosen response has gained more relative ground, the edge is positive, and the reported training loss is lower.

### What `beta` controls

Treat `beta` as a **reference-leash setting**. In reward-model RLHF, it weights the penalty for drifting from the frozen SFT model; DPO inherits that stay-close tradeoff when it compares response movement. Changing `beta` changes how strongly a given movement represents the preference, so it is not a universal quality dial. Select it with held-out preference, safety, diversity, and retention checks.

### What this training trace does not prove

The tracked example is a training pair. A better edge on it proves that the implemented objective can move in the labeled direction. It does not prove that editors prefer the model on unseen prompts.

Real preference evaluation still needs:

- held-out prompts and blinded comparisons;
- wins, losses, and ties against the accepted SFT model;
- length-balanced slices to catch shortcut learning;
- instruction, factuality, safety, and diversity gates;
- repeated runs or uncertainty estimates when the decision matters.

### DPO's practical failure modes

- **Coverage ceiling:** offline pairs cannot teach preferences they never contain.
- **Label noise:** inconsistent or weak annotators create contradictory updates.
- **Length and style shortcuts:** superficial patterns can correlate with `chosen`.
- **Distribution shift:** the trained policy may generate behavior unlike either response in the pair dataset.
- **Over-optimization:** preference improvement can trade away factuality, diversity, or instruction compliance.
- **Reference dependence:** a weak SFT anchor remains a weak starting point.

### When PPO-based RLHF remains the better tool

Choose PPO or another online RL method when the system must explore new behavior and receives meaningful feedback on generated trajectories, such as success in an interactive environment, executable tests, game outcomes, or a validated process reward. Accept the extra machinery only when that online signal provides information fixed offline pairs cannot.

Riverside's teaching run deliberately repeats one bounded proxy pair for ten steps to expose the mechanism. The 30-pair pool demonstrates data construction; production needs curated training and held-out preference sets before the broader evaluation gates in Part 3 can support adoption.


---

## Checkpoint Inventory Before Comparison

Part 1 produced three artifacts:

| Artifact | Training signal | Status |
| --- | --- | --- |
| `./checkpoints/non-instruction-full` | Raw manuscript next-token prediction | Full continued-pretraining checkpoint |
| `./checkpoints/instruction-lora` | Prompt/completion demonstrations | SFT LoRA adapter |
| `./checkpoints/preference-dpo` | One repeated bounded proxy pair | Post-SFT DPO adapter; short run remains inconclusive |

The next comparison asks whether their visible behavior matches the objective each practiced. It is not a leaderboard: the objectives, data, and parameter strategies differ.

---

## Same Questions, Four Data-Objective Checkpoints

This is a qualitative recap of the models already trained above. Every candidate receives the same three semantic questions and uses greedy decoding, so sampling noise cannot masquerade as a training effect. Candidates are loaded and released one at a time to keep memory bounded.

Read the columns as **behavior demonstrations, not a leaderboard**: continued pretraining, SFT, and DPO optimize different signals, and the single DPO pair in this teaching run is a structural proxy rather than a real editor label. Part 3 supplies the broader evaluation context.


In [ ]:
import gc
import html
import time

from IPython.display import HTML, display
from peft import PeftModel


for model_name in (
    "instruct_lora_model",
    "dpo_policy_model",
    "dpo_policy_base",
    "decoder_blocks",
    "parameter",
):
    if model_name in globals():
        del globals()[model_name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Released prior training models before loading comparison candidates.")


DATA_COMPARISON_QUESTIONS = {
    "Domain knowledge": "Who is Aria Voss, and what is her role aboard the Meridian's Promise?",
    "Instruction following": (
        "Continue the fiction narrative in the same style.\n\n"
        "Context:\nAria Voss checked the Meridian's Promise status panel and"
    ),
    "Concise editorial style": "In one concise sentence, describe the Meridian's Promise.",
}

DATA_COMPARISON_CANDIDATES = [
    ("Base", "base", MODEL_NAME),
    ("Continued pretraining", "full", "./checkpoints/non-instruction-full"),
    ("SFT LoRA", "adapter", "./checkpoints/instruction-lora"),
    ("DPO after SFT", "adapter", "./checkpoints/preference-dpo"),
]


def load_data_comparison_candidate(kind, model_path):
    """Load one comparison candidate without keeping the other checkpoints in memory."""
    if kind == "base":
        return AutoModelForCausalLM.from_pretrained(model_path).to(device)

    artifact_path = Path(model_path)
    if not artifact_path.exists():
        raise FileNotFoundError(
            f"Missing {artifact_path}. Run the corresponding training cell before this comparison."
        )
    if kind == "full":
        return AutoModelForCausalLM.from_pretrained(artifact_path).to(device)

    adapter_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
    return PeftModel.from_pretrained(adapter_base, artifact_path).to(device)


def generate_data_comparison_answer(model, question, max_new_tokens=48):
    """Use deterministic decoding so differences come from checkpoints, not sampling noise."""
    model.eval()
    formatted_prompt = render_instruction(question)
    inputs = tokenizer(formatted_prompt, return_tensors="pt")
    model_device = next(model.parameters()).device
    inputs = {name: tensor.to(model_device) for name, tensor in inputs.items()}
    prompt_length = inputs["input_ids"].shape[1]
    with torch.no_grad():
        generated = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(
        generated[0][prompt_length:], skip_special_tokens=True
    ).strip() or "[model stopped immediately]"


data_comparison_answers = {
    question_label: {} for question_label in DATA_COMPARISON_QUESTIONS
}
for candidate_label, candidate_kind, candidate_path in DATA_COMPARISON_CANDIDATES:
    candidate_model = load_data_comparison_candidate(candidate_kind, candidate_path)
    try:
        for question_label, question in DATA_COMPARISON_QUESTIONS.items():
            started = time.perf_counter()
            answer = generate_data_comparison_answer(candidate_model, question)
            elapsed = time.perf_counter() - started
            data_comparison_answers[question_label][candidate_label] = (
                f"{answer}\n\n[{elapsed:.1f}s]"
            )
    finally:
        del candidate_model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

header = "".join(
    f"<th style='min-width:190px'>{html.escape(label)}</th>"
    for label, _, _ in DATA_COMPARISON_CANDIDATES
)
body = "".join(
    "<tr>"
    f"<th style='text-align:left;vertical-align:top'>{html.escape(question_label)}</th>"
    + "".join(
        "<td style='vertical-align:top;white-space:pre-wrap'>"
        f"{html.escape(data_comparison_answers[question_label][label])}</td>"
        for label, _, _ in DATA_COMPARISON_CANDIDATES
    )
    + "</tr>"
    for question_label in DATA_COMPARISON_QUESTIONS
)
display(
    HTML(
        "<table><thead><tr><th>Same question</th>"
        + header
        + "</tr></thead><tbody>"
        + body
        + "</tbody></table>"
    )
)


---

## Optional Reference: Production Training Orchestration

The practical objective story is complete: raw manuscripts teach catalog prose, request/response pairs teach an instruction contract, and chosen/rejected pairs teach a comparative preference.

The remaining cells package those stages as separate resumable jobs with checkpointing and explicit model handoffs. Read them when you need orchestration patterns; skip to **Roadmap Checkpoint** if your goal is choosing and evaluating training objectives.

The guarded runner stays disabled because a credible production run needs more than longer training: approved data versions, clean splits, deduplication, repeated runs, independent evaluation, editor review, safety checks, serving measurements, gradual release, and rollback.

The code demonstrates stage boundaries and lineage. It does not turn the short teaching datasets into production-ready models.

In [ ]:
from pathlib import Path
import gc

import torch
from transformers import AutoModelForCausalLM, Trainer, TrainingArguments
from transformers.trainer_utils import get_last_checkpoint


def _resume_checkpoint(output_path):
    """Return the newest Trainer checkpoint in output_path, if one exists."""
    path = Path(output_path)
    return get_last_checkpoint(str(path)) if path.is_dir() else None


def _release_training_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def run_continued_pretraining(train_dataset, output_path, max_steps):
    """Continue causal-LM pretraining from a fresh base-model checkpoint."""
    output_path = str(output_path)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
    trainer = None
    try:
        args = TrainingArguments(
            output_dir=output_path,
            per_device_train_batch_size=1,
            max_steps=max_steps,
            learning_rate=5e-5,
            logging_steps=max(1, min(10, max_steps)),
            save_strategy="steps",
            save_steps=max(1, min(100, max_steps)),
            save_total_limit=2,
            report_to="none",
        )
        trainer = Trainer(model=model, args=args, train_dataset=train_dataset)
        result = trainer.train(resume_from_checkpoint=_resume_checkpoint(output_path))
        trainer.save_model(output_path)
        tokenizer.save_pretrained(output_path)
        return dict(result.metrics)
    finally:
        del trainer, model
        _release_training_memory()

### Production SFT: Versioned Adapters

Cloud training jobs usually write LoRA adapters to immutable, versioned artifact storage while keeping the base-model revision pinned separately. Promotion should require evaluation gates, checksum verification, and a rollback pointer to the previous adapter; secrets and raw training text should never be embedded in the adapter directory.

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM, Trainer, TrainingArguments


def run_lora_sft(train_dataset, output_path, max_steps):
    """Run supervised fine-tuning with a new base model and trainable LoRA adapter."""
    output_path = str(output_path)
    base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
    model = get_peft_model(
        base_model,
        LoraConfig(
            task_type=TaskType.CAUSAL_LM,
            r=8,
            lora_alpha=16,
            lora_dropout=0.05,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
            bias="none",
        ),
    )
    trainer = None
    try:
        args = TrainingArguments(
            output_dir=output_path,
            per_device_train_batch_size=1,
            max_steps=max_steps,
            learning_rate=2e-4,
            logging_steps=max(1, min(10, max_steps)),
            save_strategy="steps",
            save_steps=max(1, min(100, max_steps)),
            save_total_limit=2,
            report_to="none",
        )
        trainer = Trainer(model=model, args=args, train_dataset=train_dataset)
        result = trainer.train(resume_from_checkpoint=_resume_checkpoint(output_path))
        model.save_pretrained(output_path)
        tokenizer.save_pretrained(output_path)
        return dict(result.metrics)
    finally:
        del trainer, model, base_model
        _release_training_memory()

### Production DPO: Controlled Alignment Stage

DPO normally runs as a separate, auditable job from the promoted SFT adapter. Store the preference-dataset version, frozen-reference identity, tokenizer revision, and `beta` with the resulting adapter; gate promotion on held-out preference accuracy, safety checks, and mode-collapse tests.

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM
from trl import DPOConfig, DPOTrainer


def run_dpo(train_dataset, sft_adapter_path, output_path, max_steps):
    """Align a trainable SFT LoRA policy against an explicit frozen SFT reference."""
    sft_adapter_path = str(sft_adapter_path)
    output_path = str(output_path)
    policy_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
    reference_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
    policy = PeftModel.from_pretrained(policy_base, sft_adapter_path, is_trainable=True)
    reference = PeftModel.from_pretrained(
        reference_base, sft_adapter_path, is_trainable=False
    )
    reference.eval()
    for parameter in reference.parameters():
        parameter.requires_grad_(False)

    trainer = None
    try:
        args = DPOConfig(
            output_dir=output_path,
            per_device_train_batch_size=1,
            max_length=DPO_MAX_LENGTH,
            max_steps=max_steps,
            learning_rate=5e-5,
            beta=DPO_BETA,
            logging_steps=max(1, min(10, max_steps)),
            save_strategy="steps",
            save_steps=max(1, min(100, max_steps)),
            save_total_limit=2,
            bf16=False,
            report_to="none",
        )
        trainer = DPOTrainer(
            model=policy,
            ref_model=reference,
            args=args,
            train_dataset=train_dataset,
            processing_class=tokenizer,
        )
        result = trainer.train(resume_from_checkpoint=_resume_checkpoint(output_path))
        policy.save_pretrained(output_path)
        tokenizer.save_pretrained(output_path)
        return dict(result.metrics)
    finally:
        del trainer, policy, reference, policy_base, reference_base
        _release_training_memory()

### Production Orchestration: Build, Train, Gate, Promote

A scheduler or managed training service normally runs these stages with isolated compute, resumable checkpoints, centralized logs, and explicit data/model lineage. The guarded cell below is the local orchestration equivalent: it builds full-corpus datasets and runs the stages in dependency order, but remains off until deliberately enabled.

In [ ]:
import math

RUN_PRODUCTION_PIPELINE = False
PRODUCTION_EPOCHS = 1

production_metrics = {}
if RUN_PRODUCTION_PIPELINE:
    production_root = Path("./checkpoints/production")
    continued_path = production_root / "continued-pretraining"
    sft_path = production_root / "sft-lora"
    dpo_path = production_root / "dpo"

    # Rebuild every training contract from all mapped novels and every chapter.
    all_novel_aliases = list(NOVELS.keys())
    chapter_count = sum(
        len(list((CONTENT_DIR / directory).glob("chapter-*.txt")))
        for directory in NOVELS.values()
    )

    production_paragraphs = load_corpus_paragraphs(
        novels=all_novel_aliases, max_chapters=None
    )
    production_causal = Dataset.from_dict({"text": production_paragraphs}).map(
        lambda examples: tokenize_causal(examples, tokenizer),
        batched=True,
        remove_columns=["text"],
    )

    production_instruction_pairs = build_instruction_pairs(
        novels=all_novel_aliases, max_chapters=None
    )
    production_sft = Dataset.from_list(production_instruction_pairs).map(
        tokenize_instruction, remove_columns=["instruction", "response"]
    )

    production_preference_pairs = build_preference_pairs(
        novels=all_novel_aliases, max_chapters=None, max_pairs=None
    )
    production_dpo = Dataset.from_list(production_preference_pairs)

    print(
        f"Full corpus: {len(all_novel_aliases)} novels, {chapter_count} chapters | "
        f"continued-pretraining chunks={len(production_causal):,}, "
        f"SFT pairs={len(production_sft):,}, DPO pairs={len(production_dpo):,}"
    )

    # Derive max_steps from dataset size so each stage completes full epochs.
    continued_steps = PRODUCTION_EPOCHS * math.ceil(len(production_causal) / 1)
    sft_steps = PRODUCTION_EPOCHS * math.ceil(len(production_sft) / 1)
    dpo_steps = PRODUCTION_EPOCHS * len(production_dpo)

    production_metrics["continued_pretraining"] = run_continued_pretraining(
        train_dataset=production_causal,
        output_path=continued_path,
        max_steps=continued_steps,
    )
    production_metrics["lora_sft"] = run_lora_sft(
        train_dataset=production_sft,
        output_path=sft_path,
        max_steps=sft_steps,
    )
    production_metrics["dpo"] = run_dpo(
        train_dataset=production_dpo,
        sft_adapter_path=sft_path,
        output_path=dpo_path,
        max_steps=dpo_steps,
    )

production_metrics

---

## End of Part 1: The Capability Axis

Each objective addressed a different observed gap:

| Objective | Experience supplied | Capability targeted | Evidence still needed |
| --- | --- | --- | --- |
| Continued pretraining | Raw manuscript next-token prediction | Catalog language and house-style continuation | Clean held-out prose and retention checks |
| SFT | Prompt/completion demonstrations | Bounded instruction following | Representative contract suite |
| DPO | One repeated bounded chosen/rejected proxy pair | Relative editor preference | Held-out blinded comparisons against SFT |

The durable intuition is simple:

- raw text changes what prose the model expects;
- demonstrations change what response contract it practices;
- curated preference pairs in production change how the model ranks already-valid responses; this trace only verifies the mechanism.

These artifacts are not a quality leaderboard, and LoRA was only the practical storage choice for two local runs. Continue to **[Part 2: Parameter-Based Techniques](02-llm-finetuning-parameter-techniques.ipynb)** to open that black box and ask how much state must move. Part 3 later reconnects behavior, cost, and workload evidence.